# DOE MASTER ML PIPELINE - Equation-First V10 Chong DPHI Lock

V10 is a targeted modification of the DOE-tested V9 notebook, not a rebuild. V9 stays available as the fallback.

What V10 changes:

- locks the active runtime scope to four wells only: WellA Mallik 2L-38, WellB Mallik 5L-38, WellC Mount Elbert/MTE, and WellD IGS;
- preserves two separate input-header contracts: Mallik stacked four-row headers and Alaska MTE/IGS direct-header/refined-sheet layouts;
- makes source density porosity (`Phi_porosity`, `phi_den`, `DPHI`) the primary Chong-style `phi` feature when present;
- keeps RHOB-derived density porosity (`phi_density_calc`) as fallback/proxy evidence, not the first-choice well-log phi;
- keeps NMR-density and Archie saturation estimates on the target/review side, blocked from X predictors;
- exports row-free audit tables so the DOE run can be checked without sharing raw rows.

Source header evidence is anchored to `docs/evidence/email_screenshots_2026_06_12/` and summarized in `docs/WELL_LOG_REQUIREMENTS_MAP.md`.


## 0. Imports, paths, wells, and run policies

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable
from itertools import combinations, product
import time
import json
import math
import os
import re
import warnings

import joblib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge, SGDRegressor
from sklearn.metrics import accuracy_score, balanced_accuracy_score, brier_score_loss, f1_score, mean_absolute_error, mean_squared_error, precision_score, r2_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupKFold, ShuffleSplit
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
DOWNLOADS = Path.home() / "Downloads"
DOE_INPUT_DIR = DOWNLOADS / "Northslopedatasets06052026"
SYNTH_INPUT_DIR = Path("/mnt/data/synth_north_slope")

if DOE_INPUT_DIR.exists():
    INPUT_DIR = DOE_INPUT_DIR
    OUTPUT_DIR = DOWNLOADS / "outputs_runtime" / "ml_master"
    MODEL_DIR = DOWNLOADS / "models_runtime" / "ml_master"
elif SYNTH_INPUT_DIR.exists():
    INPUT_DIR = SYNTH_INPUT_DIR
    OUTPUT_DIR = Path("/mnt/data/synth_workspace/outputs_runtime/ml_master")
    MODEL_DIR = Path("/mnt/data/synth_workspace/models_runtime/ml_master")
else:
    INPUT_DIR = DOE_INPUT_DIR
    OUTPUT_DIR = DOWNLOADS / "outputs_runtime" / "ml_master"
    MODEL_DIR = DOWNLOADS / "models_runtime" / "ml_master"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
CODE_VERSION = "V10_chong_dphi_lock"
RUN_SLUG = CODE_VERSION.lower()

ACTIVE_PROJECT_WELLS = ["WellA", "WellB", "WellC", "WellD"]
ACTIVE_PROJECT_WELL_NAMES = {
    "WellA": "Mallik 2L-38",
    "WellB": "Mallik 5L-38",
    "WellC": "Mount Elbert / MTE",
    "WellD": "IGS",
}
ACTIVE_WELL_SCOPE_NOTE = "Runtime uses exactly the four active project wells; no fifth literature-only well is loaded or synthesized."
LITERATURE_SCOPE_NOTE = "Chong et al. provides the comparison workflow, but V10 uses only the active four-well project data available in this runtime."
DENSITY_POROSITY_SOURCE_POLICY = (
    "Use source density-porosity headers (Phi_porosity, phi_den, DPHI) as Chong phi when present; "
    "use RHOB-derived phi_D only as fallback/proxy review evidence."
)
SOURCE_HEADER_EVIDENCE_NOTE = (
    "Mallik headers come from 2026-06-05 stacked Excel header screenshots; "
    "Alaska MTE/IGS headers and refined target layouts come from 2026-06-08 screenshots."
)
HEADER_CONTRACTS = {
    "mallik_stacked_2026_06_05": {
        "layout": "mallik_stacked",
        "scope": "Mallik 2L-38 and Mallik 5L-38 stacked four-row header workbook",
        "screenshots": [
            "docs/evidence/email_screenshots_2026_06_12/screenshot_2026-06-05_131418.png",
            "docs/evidence/email_screenshots_2026_06_12/screenshot_2026-06-05_131426.png",
        ],
        "visible_headers": ["DEPTH", "Rho_b", "Phi_porosity", "Differential Caliper", "Deep formation resistivity", "GR", "Vs", "Vp", "Ratio Vp/Vs", "Impedance", "Sgh", "NMR_SAT"],
    },
    "alaska_mte_direct_2026_06_08": {
        "layout": "direct_header",
        "scope": "MTE direct-header sheet plus MTE_refined target/alignment sheet",
        "screenshots": [
            "docs/evidence/email_screenshots_2026_06_12/screenshot_2026-06-08_111056.png",
            "docs/evidence/email_screenshots_2026_06_12/screenshot_2026-06-08_111117.png",
        ],
        "visible_headers": ["Depth_ft", "Density_gpcc", "phi_den", "phi_nmr", "S_h", "S_wr", "GR", "phi_neut", "CAL1", "AO90", "VELP", "VS1"],
    },
    "alaska_igs_direct_2026_06_08": {
        "layout": "direct_header",
        "scope": "IGS direct-header sheet plus IGS_refined target/alignment sheet",
        "screenshots": [
            "docs/evidence/email_screenshots_2026_06_12/screenshot_2026-06-08_111108.png",
            "docs/evidence/email_screenshots_2026_06_12/screenshot_2026-06-08_111124.png",
        ],
        "visible_headers": ["DEPT", "RHOB", "NPHI", "DPHI", "NMRPHI", "GR", "caliper", "RES", "VP", "VS", "Sh", "Swr"],
    },
}


# -----------------------------------------------------------------------------
# Well metadata and target map
# -----------------------------------------------------------------------------
# NOTE: latitude/longitude are kept as context only. They are not ML predictors.
# Site/well identity is used only to choose equation assumptions such as Archie a/m/n.
WELL_METADATA: dict[str, dict[str, Any]] = {
    "WellA": {
        "well_name": "Mallik 2L-38",
        "site": "Mallik / Mackenzie Delta analogue",
        "file_contains": ["2L-38"],
        "sheet": None,
        "layout": "mallik_stacked",
        "source_header_contract": "mallik_stacked_2026_06_05",
        "lat": np.nan,
        "lon": np.nan,
        "archie_a": np.nan,
        "archie_m": np.nan,
        "archie_n": np.nan,
        "archie_note": "Mallik uses NMR-density saturation as primary reference; Archie constants not forced unless approved.",
        "hydrate_target_header": "Sgh",
        "water_target_header": None,
    },
    "WellB": {
        "well_name": "Mallik 5L-38",
        "site": "Mallik / Mackenzie Delta analogue",
        "file_contains": ["5L-38"],
        "sheet": None,
        "layout": "mallik_stacked",
        "source_header_contract": "mallik_stacked_2026_06_05",
        "lat": np.nan,
        "lon": np.nan,
        "archie_a": np.nan,
        "archie_m": np.nan,
        "archie_n": np.nan,
        "archie_note": "Mallik uses NMR-density saturation as primary reference; Archie constants not forced unless approved.",
        "hydrate_target_header": "Sgh",
        "water_target_header": None,
    },
    "WellC": {
        "well_name": "Mount Elbert Stratigraphic Test Well",
        "site": "Mount Elbert / Eileen Gas Hydrate Trend",
        "file_contains": ["MtElbert", "Ignik", "ANS"],
        "sheet": "MTE",
        "layout": "direct_header",
        "source_header_contract": "alaska_mte_direct_2026_06_08",
        "lat": np.nan,
        "lon": np.nan,
        "archie_a": 1.0,
        "archie_m": 1.9,
        "archie_n": 2.0,
        "archie_note": "Alaska calibration window: Mount Elbert a=1.0, m=1.9, n=2.0.",
        "hydrate_target_header": "S_h",
        "water_target_header": "S_wr",
    },
    "WellD": {
        "well_name": "Iġnik Sikumi Test Well",
        "site": "Iġnik Sikumi / Eileen Gas Hydrate Trend",
        "file_contains": ["MtElbert", "Ignik", "ANS"],
        "sheet": "IGS",
        "layout": "direct_header",
        "source_header_contract": "alaska_igs_direct_2026_06_08",
        "lat": np.nan,
        "lon": np.nan,
        "archie_a": 1.6,
        "archie_m": 2.0,
        "archie_n": 2.0,
        "archie_note": "Alaska calibration window: Iġnik Sikumi a=1.6, m=2.0, n=2.0.",
        "hydrate_target_header": "Sh",
        "water_target_header": "Swr",
    },
}

# -----------------------------------------------------------------------------
# Equation constants and run policies
# -----------------------------------------------------------------------------
DENSITY_MATRIX_G_CC = 2.65
DENSITY_FLUID_G_CC = 1.02
SURFACE_PRESSURE_MPA = 0.101325
WATER_DENSITY_KG_M3 = 1020.0
GRAVITY_M_S2 = 9.80665

# Rickman-style brittleness normalization constants. Young's modulus is first
# converted from GPa to Mpsi so the mentor equation bounds (1 to 8 Mpsi) are
# applied in the expected units.
GPA_PER_MPSI = 6.894757293168361
BRITTLE_YOUNGS_MIN_MPSI = 1.0
BRITTLE_YOUNGS_MAX_MPSI = 8.0
BRITTLE_PR_MIN = 0.15
BRITTLE_PR_MAX = 0.40
CLIP_BRITTLENESS_TO_0_100 = True

# Current data note from project review: workbook values are normalized except depth.
# V10 keeps that caveat explicit and retains the full mentor/Rickman geomechanical
# equation set to the ML experiment. Equation features are fully integrated as normalized/proxy
# predictors where allowed, while target-derived saturation fields remain blocked.
NORMALIZED_INPUT_MODE = True
PRIMARY_FEATURE_POLICY = "chong_ann_core_plus_equation_integrated"
ALLOW_PROXY_EQUATION_FEATURES_IN_PRIMARY_MODEL = True
ALLOW_PRESSURE_CONTEXT_IN_PRIMARY_MODEL = False
EVALUATE_PROXY_FEATURE_SET_FOR_REVIEW = True

# Resistivity has been confirmed by project review as the deep formation resistivity family.
DEEP_RESISTIVITY_ALIAS_CONFIRMED = True
DEEP_RESISTIVITY_ALIASES_CONFIRMED = ["AO90", "A090", "AF90", "Deep formation resistivity", "Apparent Resistivity", "RES"]

# Caliper review: project notes say caliper is expected around 7.5-8.5 if physical
# diameter is present. If the caliper curve is normalized, the code marks it as
# normalized/context-only rather than failing the interval.
CALIPER_EXPECTED_MIN = 7.5
CALIPER_EXPECTED_MAX = 8.5

# If Rw is not supplied, estimate from S_wr/Swr where available.
# In normalized-input mode, this is a relative Archie-review proxy, not a final physical Rw.
ESTIMATE_RW_FROM_WATER_TARGET = True
RW_TRIM_QUANTILES = (0.05, 0.95)

# Single-well transfer design for current mentor review.
# Train on one known/calibration well, then evaluate on the other wells as external transfer wells.
CALIBRATION_TRAIN_WELL = "WellC"  # Mount Elbert Stratigraphic Test Well
TRANSFER_TEST_WELLS = ["WellA", "WellB", "WellD"]

HYDRATE_TRAIN_WELLS = [CALIBRATION_TRAIN_WELL]
HYDRATE_VALIDATION_WELLS = TRANSFER_TEST_WELLS.copy()

OCCURRENCE_TRAIN_WELLS = [CALIBRATION_TRAIN_WELL]
OCCURRENCE_VALIDATION_WELLS = TRANSFER_TEST_WELLS.copy()
OCCURRENCE_POSITIVE_SH_THRESHOLD = 0.05
OCCURRENCE_NEGATIVE_SH_THRESHOLD = 0.01
OCCURRENCE_CLASSIFICATION_THRESHOLD = 0.50

# Water model: only wells with water target currently available will survive target filtering.
# With the current workbooks this is mostly WellC -> WellD, so water remains diagnostic-only.
WATER_TRAIN_WELLS = [CALIBRATION_TRAIN_WELL]
WATER_VALIDATION_WELLS = TRANSFER_TEST_WELLS.copy()
WATER_MODEL_MODE = "diagnostic_only"

MIN_TRAIN_FEATURE_COVERAGE = 0.35
MIN_ROW_FEATURE_FRACTION = 0.40

# Keep NMR porosity out of the default ML predictors because NMR-density saturation
# may be a target/calibration reference. The equation is still calculated for review.
INCLUDE_NMR_POROSITY_AS_FEATURE = False

# Accuracy/correctness controls.
# With one training well, model selection must happen inside that well.
# We use contiguous depth-block CV instead of selecting from the external transfer wells.
USE_GROUP_CV_INSIDE_TRAINING_WELLS = False
TRAINING_SELECTION_MODE = "single_train_well_depth_block_cv"
DEPTH_BLOCK_CV_N_SPLITS = 5
APPLY_TARGET_BIN_WEIGHTS = True
SATURATION_BIN_EDGES = [0.0, 0.01, 0.05, 0.20, 0.50, 1.0000001]
SATURATION_BIN_LABELS = ["zero_to_trace", "trace_to_occurrence", "low", "moderate", "high"]
REPORT_BLIND_WELL_ONLY_AFTER_CV_SELECTION = True


# -----------------------------------------------------------------------------
# Chong et al.-style ANN sensitivity controls
# -----------------------------------------------------------------------------
# "full_sensitivity" is the paper-style run: broad hyperparameter sensitivity and
# 100 ANN realizations. It may take a long time on a CPU. For a smoke test, set
# the environment variable CHONG_RUNTIME_MODE=quick_test before running.
CHONG_RUNTIME_MODE = os.environ.get("CHONG_RUNTIME_MODE", "full_sensitivity")
CHONG_ANN_REALIZATIONS = 100 if CHONG_RUNTIME_MODE == "full_sensitivity" else 5
CHONG_TUNING_SPLITS = 5 if CHONG_RUNTIME_MODE == "full_sensitivity" else 2
CHONG_WLC_MODE = "all_single_pair_triplet" if CHONG_RUNTIME_MODE == "full_sensitivity" else "reported_core"
CHONG_RUN_BASIN_TRANSFER_ANALOGS = True
CHONG_RUN_CURRENT_SINGLE_WELL_TRANSFER = True
CHONG_RUN_EQUATION_INTEGRATED_EXTENSION = True
CHONG_SAVE_TOP_PREDICTION_TABLES = 12

# Hyperparameter ranges follow Chong et al.'s reported broad tuning ranges.
CHONG_GRID_LEARNING_RATES = [0.0001, 0.0003, 0.001, 0.003, 0.01]
CHONG_GRID_HIDDEN_LAYERS = [2, 3]
CHONG_GRID_NODES_PER_LAYER = [10, 20, 30, 40, 50]
CHONG_GRID_BATCH_SIZES = [100, 250, 500]
CHONG_GRID_EPOCHS = [100, 250, 500]
CHONG_GRID_DROPOUT = [0.5]
CHONG_GRID_SAMPLE_LIMIT = None if CHONG_RUNTIME_MODE == "full_sensitivity" else 8

# Fixed architecture reported as optimal by Chong et al. for both ANS-trained and
# Mallik-trained cases: two hidden layers, 40 nodes/layer, learning rate 0.001,
# batch size 100, 500 epochs, ReLU hidden layers, linear output, dropout 0.5.
CHONG_FIXED_ANN_PARAMS = {
    "hidden_layers": 2,
    "nodes_per_layer": 40,
    "learning_rate": 0.001,
    "batch_size": 100,
    "epochs": 500,
    "dropout": 0.5,
}

# If a full run is too slow on a local machine, temporarily switch to:
#   import os; os.environ["CHONG_RUNTIME_MODE"] = "quick_test"
# then restart the kernel and rerun.

print("RUN_ID:", RUN_ID)
print("CODE_VERSION:", CODE_VERSION)
print("ACTIVE_PROJECT_WELLS:", ACTIVE_PROJECT_WELLS)
print("DENSITY_POROSITY_SOURCE_POLICY:", DENSITY_POROSITY_SOURCE_POLICY)
print("SOURCE_HEADER_EVIDENCE_NOTE:", SOURCE_HEADER_EVIDENCE_NOTE)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("PRIMARY_FEATURE_POLICY:", PRIMARY_FEATURE_POLICY)
print("WATER_MODEL_MODE:", WATER_MODEL_MODE)
print("SINGLE_WELL_TRANSFER:", f"train {CALIBRATION_TRAIN_WELL} -> test {'+'.join(TRANSFER_TEST_WELLS)}")

print("CHONG_RUNTIME_MODE:", CHONG_RUNTIME_MODE)
print("CHONG_ANN_REALIZATIONS:", CHONG_ANN_REALIZATIONS)
print("CHONG_WLC_MODE:", CHONG_WLC_MODE)


## 1. Load workbooks and map raw headers to canonical fields

In [ ]:

# =============================================================================
# Workbook loading, standardization, and source provenance
# =============================================================================

def normalize_token(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return "".join(ch for ch in str(value).strip().lower() if ch.isalnum())


def clean_text(value: Any, fallback: str = "") -> str:
    try:
        missing = pd.isna(value)
    except Exception:
        missing = False
    if missing:
        return fallback
    text = re.sub(r"\s+", " ", str(value).strip())
    return text or fallback


def numeric_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def normalize_fraction(s: pd.Series) -> pd.Series:
    x = numeric_series(s)
    med = x.dropna().median() if x.notna().any() else np.nan
    if pd.notna(med) and med > 1.5:
        x = x / 100.0
    return x.clip(lower=0.0, upper=1.0)


def normalize_density_gcc(s: pd.Series) -> pd.Series:
    x = numeric_series(s)
    med = x.dropna().median() if x.notna().any() else np.nan
    # kg/m3 -> g/cc
    if pd.notna(med) and med > 20:
        x = x / 1000.0
    return x


def normalize_velocity_ms(s: pd.Series) -> pd.Series:
    x = numeric_series(s)
    med = x.dropna().median() if x.notna().any() else np.nan
    # km/s -> m/s
    if pd.notna(med) and 0.5 < med < 15:
        x = x * 1000.0
    return x


def find_input_file(tokens: list[str]) -> Path:
    files = [p for p in INPUT_DIR.glob("*.xls*") if not p.name.startswith("~$")]
    for token in tokens:
        for p in files:
            if token.lower() in p.name.lower():
                return p
    raise FileNotFoundError(f"Could not find workbook containing any of {tokens} under {INPUT_DIR}")


def read_mallik_stacked(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw = pd.read_excel(path, sheet_name=0, header=None)
    if raw.shape[0] < 5:
        raise ValueError(f"{path.name} does not look like a stacked Mallik workbook")
    role_row = raw.iloc[0]
    header_row = raw.iloc[1]
    unit_row = raw.iloc[2]
    desc_row = raw.iloc[3]
    headers = []
    provenance_rows = []
    for i in range(raw.shape[1]):
        header = clean_text(header_row.iloc[i], f"column_{i}")
        if header.lower().startswith("unnamed") or header == "":
            header = clean_text(role_row.iloc[i], f"column_{i}")
        base = header
        count = sum(1 for h in headers if h == base or h.startswith(base + "."))
        if count:
            header = f"{base}.{count}"
        headers.append(header)
        provenance_rows.append({
            "source_column_index": i,
            "source_header": header,
            "role_row": clean_text(role_row.iloc[i]),
            "mnemonic_row": clean_text(header_row.iloc[i]),
            "unit_row": clean_text(unit_row.iloc[i]),
            "description_row": clean_text(desc_row.iloc[i]),
        })
    df = raw.iloc[4:].copy()
    df.columns = headers
    df = df.dropna(how="all").reset_index(drop=True)
    return df, pd.DataFrame(provenance_rows)


def read_direct_header(path: Path, sheet: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.read_excel(path, sheet_name=sheet)
    df = df.dropna(how="all").reset_index(drop=True)
    prov = pd.DataFrame([
        {
            "source_column_index": i,
            "source_header": c,
            "role_row": "",
            "mnemonic_row": c,
            "unit_row": "",
            "description_row": "",
        }
        for i, c in enumerate(df.columns)
    ])
    return df, prov


def read_raw_well(alias: str, meta: dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame, Path]:
    path = find_input_file(meta["file_contains"])
    if meta["layout"] == "mallik_stacked":
        df, prov = read_mallik_stacked(path)
    else:
        df, prov = read_direct_header(path, meta["sheet"])
    prov.insert(0, "well_alias", alias)
    prov.insert(1, "well_name", meta["well_name"])
    prov.insert(2, "source_workbook", path.name)
    prov.insert(3, "source_sheet", meta.get("sheet") or "first_sheet")
    prov.insert(4, "source_layout", meta["layout"])
    prov.insert(5, "source_header_contract", meta.get("source_header_contract", ""))
    return df, prov, path


def get_column(df: pd.DataFrame, aliases: Iterable[str]) -> tuple[pd.Series | None, str | None]:
    norm_to_col = {normalize_token(c): c for c in df.columns}
    aliases_norm = [normalize_token(a) for a in aliases]
    # exact first
    for a in aliases_norm:
        if a in norm_to_col:
            col = norm_to_col[a]
            return df[col], col
    # token containment second
    for a in aliases_norm:
        for norm, col in norm_to_col.items():
            if a and (a in norm or norm in a):
                return df[col], col
    return None, None


FIELD_ALIASES: dict[str, list[str]] = {
    "depth": ["depth_m", "depth", "true depth", "true_depth", "Depth_ft", "DEPT", "MD", "TVD"],
    "rhob_g_cc": ["rhob", "rho_b", "density_gpcc", "density_gcpcc", "density", "bulk density", "Rho_b", "RHOB"],
    "density_porosity_vv": ["phi_den", "phi_porosity", "density porosity", "density_porosity", "DPHI", "PHID", "Phi_porosity"],
    "nmr_porosity_vv": ["phi_nmr", "nmrphi", "NMRPHI", "nmr porosity", "NMR_POR", "CMR_PHI"],
    "neutron_porosity_vv": ["phi_neut", "neutron porosity", "NPHI", "TNPH"],
    "gr_api": ["gr", "gamma ray", "gamma_ray", "GR"],
    "rt_ohm_m": ["rt", "res", "resistivity", "deep formation resistivity", "A090", "AO90", "AF90", "ILD", "LLD"],
    "vp_m_s": ["vp", "velp", "compressional wave velocity", "p wave", "dtc_velocity"],
    "vs_m_s": ["vs", "vs1", "shear wave velocity", "s wave", "dts_velocity"],
    "caliper_raw": ["caliper", "cal1", "differential caliper", "cali"],
}


def standardize_well(alias: str, raw: pd.DataFrame, prov: pd.DataFrame, meta: dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = pd.DataFrame(index=raw.index)
    mapping_rows = []

    def assign_field(field: str, transform=None):
        ser, col = get_column(raw, FIELD_ALIASES[field])
        if ser is None:
            out[field] = np.nan
            mapping_rows.append({"well_alias": alias, "canonical_field": field, "source_header": None, "coverage": 0.0, "note": "not_found"})
            return
        out[field] = transform(ser) if transform else numeric_series(ser)
        note = "mapped"
        if field == "rt_ohm_m" and DEEP_RESISTIVITY_ALIAS_CONFIRMED:
            note = "mapped_confirmed_deep_formation_resistivity"
        elif field == "caliper_raw":
            note = "mapped_caliper_qc_context"
        mapping_rows.append({
            "well_alias": alias,
            "canonical_field": field,
            "source_header": col,
            "coverage": float(out[field].notna().mean()),
            "note": note,
        })

    # Depth can be ft or m based on source header. Keep both for context.
    depth_ser, depth_col = get_column(raw, FIELD_ALIASES["depth"])
    if depth_ser is None:
        out["depth_m"] = np.nan
        out["depth_ft"] = np.nan
        mapping_rows.append({"well_alias": alias, "canonical_field": "depth_m", "source_header": None, "coverage": 0.0, "note": "not_found"})
    else:
        depth_raw = numeric_series(depth_ser)
        depth_norm = normalize_token(depth_col)
        if "ft" in depth_norm:
            out["depth_ft"] = depth_raw
            out["depth_m"] = depth_raw * 0.3048
            note = "converted_ft_to_m"
        else:
            out["depth_m"] = depth_raw
            out["depth_ft"] = depth_raw / 0.3048
            note = "assumed_m"
        mapping_rows.append({"well_alias": alias, "canonical_field": "depth_m", "source_header": depth_col, "coverage": float(out["depth_m"].notna().mean()), "note": note})

    assign_field("rhob_g_cc", normalize_density_gcc)
    assign_field("density_porosity_vv", normalize_fraction)
    assign_field("nmr_porosity_vv", normalize_fraction)
    assign_field("neutron_porosity_vv", normalize_fraction)
    assign_field("gr_api", numeric_series)
    assign_field("rt_ohm_m", numeric_series)
    assign_field("vp_m_s", normalize_velocity_ms)
    assign_field("vs_m_s", normalize_velocity_ms)
    assign_field("caliper_raw", numeric_series)

    # Targets by exact approved header map.
    h_ser, h_col = get_column(raw, [meta["hydrate_target_header"]])
    out["hydrate_saturation_reference"] = normalize_fraction(h_ser) if h_ser is not None else np.nan
    mapping_rows.append({"well_alias": alias, "canonical_field": "hydrate_saturation_reference", "source_header": h_col, "coverage": float(pd.Series(out["hydrate_saturation_reference"]).notna().mean()), "note": "target" if h_col else "not_found"})

    water_header = meta.get("water_target_header")
    if water_header:
        w_ser, w_col = get_column(raw, [water_header])
        out["water_saturation_reference"] = normalize_fraction(w_ser) if w_ser is not None else np.nan
        note = "target" if w_col else "not_found"
    else:
        w_col = None
        out["water_saturation_reference"] = np.nan
        note = "not_applicable"
    mapping_rows.append({"well_alias": alias, "canonical_field": "water_saturation_reference", "source_header": w_col, "coverage": float(pd.Series(out["water_saturation_reference"]).notna().mean()), "note": note})

    # Well metadata context.
    out.insert(0, "well_alias", alias)
    out.insert(1, "well_name", meta["well_name"])
    out.insert(2, "site", meta["site"])
    out["lat"] = meta.get("lat", np.nan)
    out["lon"] = meta.get("lon", np.nan)
    out["archie_a"] = meta.get("archie_a", np.nan)
    out["archie_m"] = meta.get("archie_m", np.nan)
    out["archie_n"] = meta.get("archie_n", np.nan)
    out["source_hydrate_header"] = meta["hydrate_target_header"]
    out["source_water_header"] = meta.get("water_target_header")
    out["source_layout"] = meta["layout"]
    out["source_header_contract"] = meta.get("source_header_contract", "")

    # Clean invalid resistivity and velocities.
    out.loc[out["rt_ohm_m"] <= 0, "rt_ohm_m"] = np.nan
    out.loc[out["vp_m_s"] <= 0, "vp_m_s"] = np.nan
    out.loc[out["vs_m_s"] <= 0, "vs_m_s"] = np.nan

    map_df = pd.DataFrame(mapping_rows)
    map_df.insert(1, "well_name", meta["well_name"])
    map_df["source_layout"] = meta["layout"]
    map_df["source_header_contract"] = meta.get("source_header_contract", "")
    contract = HEADER_CONTRACTS.get(meta.get("source_header_contract", ""), {})
    map_df["source_contract_screenshots"] = "; ".join(contract.get("screenshots", []))
    return out, map_df


def load_all_wells() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    frames = []
    mapping = []
    provenance = []
    for alias, meta in WELL_METADATA.items():
        raw, prov, path = read_raw_well(alias, meta)
        std, map_df = standardize_well(alias, raw, prov, meta)
        std["source_workbook"] = path.name
        std["source_sheet"] = meta.get("sheet") or "first_sheet"
        frames.append(std)
        mapping.append(map_df)
        provenance.append(prov)
        print(f"Loaded {alias}: {meta['well_name']} from {path.name}, rows={len(std)}")
    return pd.concat(frames, ignore_index=True), pd.concat(mapping, ignore_index=True), pd.concat(provenance, ignore_index=True)

standardized, field_mapping, source_columns = load_all_wells()
print("Combined rows:", len(standardized))
display(field_mapping)
print("Source header contracts:")
display(source_columns[["well_alias", "well_name", "source_workbook", "source_sheet", "source_layout", "source_header_contract"]].drop_duplicates())


## 2. Compute equation-derived fields before ML

In [ ]:

# =============================================================================
# Equation-first feature engineering
# =============================================================================

def clip01(x: pd.Series | np.ndarray | float) -> pd.Series | np.ndarray | float:
    return np.clip(x, 0.0, 1.0)


def safe_divide(numer, denom):
    return np.where(np.abs(denom) > 1e-12, numer / denom, np.nan)


def estimate_rw_by_well(df: pd.DataFrame) -> dict[str, float]:
    rw_by_well: dict[str, float] = {}
    for alias, group in df.groupby("well_alias"):
        meta = WELL_METADATA[alias]
        a = meta.get("archie_a", np.nan)
        m = meta.get("archie_m", np.nan)
        n = meta.get("archie_n", np.nan)
        if not np.isfinite(a) or not np.isfinite(m) or not np.isfinite(n):
            rw_by_well[alias] = np.nan
            continue
        if not ESTIMATE_RW_FROM_WATER_TARGET or group["water_saturation_reference"].notna().sum() < 10:
            rw_by_well[alias] = np.nan
            continue
        phi = group["density_porosity_vv"].copy()
        phi = phi.fillna(group.get("phi_density_calc", pd.Series(index=group.index, dtype=float)))
        rt = group["rt_ohm_m"]
        sw = group["water_saturation_reference"].clip(0.02, 0.98)
        rw = (sw ** n) * rt * (phi ** m) / a
        valid = rw.replace([np.inf, -np.inf], np.nan).dropna()
        valid = valid[(valid > 0) & (valid < valid.quantile(0.99) * 1.5)]
        if len(valid) >= 10:
            lo, hi = valid.quantile(RW_TRIM_QUANTILES[0]), valid.quantile(RW_TRIM_QUANTILES[1])
            valid = valid[(valid >= lo) & (valid <= hi)]
            rw_by_well[alias] = float(valid.median())
        else:
            rw_by_well[alias] = np.nan
    return rw_by_well


def add_equation_features(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df.copy()
    equations = []

    # Hydrostatic pressure.
    pressure_gradient = WATER_DENSITY_KG_M3 * GRAVITY_M_S2 / 1_000_000.0
    out["pressure_hydrostatic_mpa"] = SURFACE_PRESSURE_MPA + pressure_gradient * out["depth_m"]
    equations.append({"field": "pressure_hydrostatic_mpa", "equation": "P_abs = P_surface + rho_w*g*z", "role": "context", "predictor_allowed": True, "notes": "Hydrostatic screening pressure only; not hydrate proof."})

    # Density porosity.
    out["phi_density_calc"] = (DENSITY_MATRIX_G_CC - out["rhob_g_cc"]) / (DENSITY_MATRIX_G_CC - DENSITY_FLUID_G_CC)
    out["phi_density_calc"] = out["phi_density_calc"].clip(0.0, 0.7)
    equations.append({"field": "phi_density_calc", "equation": "phi_D = (rho_matrix - rho_bulk) / (rho_matrix - rho_fluid)", "role": "feature", "predictor_allowed": True, "notes": f"rho_matrix={DENSITY_MATRIX_G_CC} g/cc; rho_fluid={DENSITY_FLUID_G_CC} g/cc."})

    # Use source density porosity when available, otherwise calculated density porosity.
    source_phi_available = out["density_porosity_vv"].notna()
    fallback_phi_available = out["phi_density_calc"].notna()
    phi = out["density_porosity_vv"].copy()
    phi = phi.fillna(out["phi_density_calc"])
    out["phi_effective_for_equations"] = phi.clip(0.001, 0.7)
    out["phi_effective_source"] = np.where(
        source_phi_available,
        "source_density_porosity_header",
        np.where(fallback_phi_available, "rhob_density_equation_fallback", "missing"),
    )
    equations.append({"field": "phi_effective_for_equations", "equation": "preferred phi = source density porosity else phi_D", "role": "feature", "predictor_allowed": True, "notes": DENSITY_POROSITY_SOURCE_POLICY})
    equations.append({"field": "phi_effective_source", "equation": "source flag for phi_effective_for_equations", "role": "audit", "predictor_allowed": False, "notes": "Audit flag only; not a numeric ML feature."})

    # NMR-density hydrate saturation.
    out["sh_nmr_density_calc"] = ((out["phi_effective_for_equations"] - out["nmr_porosity_vv"]) / out["phi_effective_for_equations"]).replace([np.inf, -np.inf], np.nan).clip(0.0, 1.0)
    equations.append({"field": "sh_nmr_density_calc", "equation": "Sh_NMRDEN = (phi - phi_NMR) / phi", "role": "target/calibration review", "predictor_allowed": False, "notes": "Kept out of X_allowed when learning hydrate saturation."})

    # GR shale volume proxy: simple linear index plus Larionov Tertiary form.
    gr_min = out["gr_api"].quantile(0.05)
    gr_max = out["gr_api"].quantile(0.95)
    if pd.isna(gr_min) or pd.isna(gr_max) or gr_max <= gr_min:
        gr_min, gr_max = 20.0, 120.0
    out["gr_index"] = ((out["gr_api"] - gr_min) / (gr_max - gr_min)).clip(0.0, 1.0)
    out["vsh_larionov_tertiary"] = (0.083 * (2 ** (3.7 * out["gr_index"]) - 1)).clip(0.0, 1.0)
    out["clean_sand_score"] = (1.0 - out["vsh_larionov_tertiary"]).clip(0.0, 1.0)
    equations.append({"field": "vsh_larionov_tertiary", "equation": "Vsh = 0.083*(2^(3.7*IGR)-1)", "role": "feature/context", "predictor_allowed": True, "notes": "Screening proxy using dataset 5th/95th GR anchors."})

    # Resistivity transform.
    out["log10_rt"] = np.log10(out["rt_ohm_m"].where(out["rt_ohm_m"] > 0))
    equations.append({"field": "log10_rt", "equation": "log10(Rt)", "role": "feature", "predictor_allowed": True, "notes": "Resistivity transform for ML."})

    # Elastic and mentor/Rickman-style geomechanical attributes.
    rho_kg_m3 = out["rhob_g_cc"] * 1000.0
    vp = out["vp_m_s"]
    vs = out["vs_m_s"]
    vp2 = vp ** 2
    vs2 = vs ** 2
    elastic_denominator = (vp2 - vs2).replace(0, np.nan)

    out["vp_vs_ratio"] = (vp / vs).replace([np.inf, -np.inf], np.nan)
    out["acoustic_impedance"] = out["rhob_g_cc"] * vp
    out["shear_impedance"] = out["rhob_g_cc"] * vs
    out["shear_modulus_gpa"] = (rho_kg_m3 * vs2) / 1e9
    out["bulk_modulus_gpa"] = (rho_kg_m3 * (vp2 - (4.0 / 3.0) * vs2)) / 1e9

    # Dynamic Young's modulus and Poisson's ratio follow the mentor screenshot
    # equations directly from density, Vp, and Vs. The Young's-modulus formula is
    # algebraically equivalent to 9KG/(3K+G), but keeping the direct form makes the
    # equation audit match the mentor geomechanics reference.
    out["youngs_modulus_gpa"] = (rho_kg_m3 * vs2 * ((3.0 * vp2) - (4.0 * vs2)) / elastic_denominator) / 1e9
    out["youngs_modulus_mpsi"] = out["youngs_modulus_gpa"] / GPA_PER_MPSI
    out["poisson_ratio"] = ((vp2 - 2.0 * vs2) / (2.0 * elastic_denominator)).replace([np.inf, -np.inf], np.nan)

    out["lambda_gpa"] = out["bulk_modulus_gpa"] - (2.0 / 3.0) * out["shear_modulus_gpa"]
    out["mu_gpa"] = out["shear_modulus_gpa"]
    out["lambda_rho"] = out["lambda_gpa"] * out["rhob_g_cc"]
    out["mu_rho"] = out["mu_gpa"] * out["rhob_g_cc"]

    # Mentor impedance-form lambda-rho / mu-rho checks. These are retained as
    # separate terms because the screenshot writes MR = SI^2 and LR = AI^2 - 2*SI^2.
    out["lr_term"] = (out["acoustic_impedance"] ** 2) - 2.0 * (out["shear_impedance"] ** 2)
    out["mr_term"] = out["shear_impedance"] ** 2

    out["brittleness_youngs_pct"] = ((out["youngs_modulus_mpsi"] - BRITTLE_YOUNGS_MIN_MPSI) / (BRITTLE_YOUNGS_MAX_MPSI - BRITTLE_YOUNGS_MIN_MPSI)) * 100.0
    out["brittleness_poisson_pct"] = ((out["poisson_ratio"] - BRITTLE_PR_MAX) / (BRITTLE_PR_MIN - BRITTLE_PR_MAX)) * 100.0
    if CLIP_BRITTLENESS_TO_0_100:
        out["brittleness_youngs_pct"] = out["brittleness_youngs_pct"].clip(0.0, 100.0)
        out["brittleness_poisson_pct"] = out["brittleness_poisson_pct"].clip(0.0, 100.0)
    out["brittleness_total_pct"] = (out["brittleness_youngs_pct"] + out["brittleness_poisson_pct"]) / 2.0

    equations.extend([
        {"field": "vp_vs_ratio", "equation": "Rv = Vp / Vs", "role": "feature", "predictor_allowed": True, "notes": "Elastic ratio feature."},
        {"field": "acoustic_impedance", "equation": "AI = rho_bulk * Vp", "role": "feature", "predictor_allowed": True, "notes": "P-wave impedance / stiffness-sensitive property."},
        {"field": "shear_impedance", "equation": "SI = rho_bulk * Vs", "role": "feature", "predictor_allowed": True, "notes": "S-wave impedance term used by MR and LR."},
        {"field": "shear_modulus_gpa", "equation": "G = rho * Vs^2", "role": "feature", "predictor_allowed": True, "notes": "Computed in GPa; equivalent to mu."},
        {"field": "bulk_modulus_gpa", "equation": "K = rho*(Vp^2 - 4/3*Vs^2)", "role": "feature", "predictor_allowed": True, "notes": "Computed in GPa."},
        {"field": "youngs_modulus_gpa", "equation": "YM = rho*Vs^2*(3*Vp^2 - 4*Vs^2)/(Vp^2 - Vs^2)", "role": "feature", "predictor_allowed": True, "notes": "Dynamic Young's modulus; mentor geomechanics equation."},
        {"field": "youngs_modulus_mpsi", "equation": "YM_Mpsi = YM_GPa / 6.894757", "role": "feature", "predictor_allowed": True, "notes": "Unit conversion for Rickman brittleness normalization."},
        {"field": "poisson_ratio", "equation": "PR = (Vp^2 - 2*Vs^2)/(2*(Vp^2 - Vs^2))", "role": "feature", "predictor_allowed": True, "notes": "Dynamic Poisson's ratio; mentor geomechanics equation."},
        {"field": "lambda_rho", "equation": "lambda_rho = (K - 2/3G) * rho", "role": "feature", "predictor_allowed": True, "notes": "Fluid/incompressibility-sensitive elastic feature."},
        {"field": "mu_rho", "equation": "mu_rho = G * rho", "role": "feature", "predictor_allowed": True, "notes": "Rigidity-sensitive elastic feature."},
        {"field": "lr_term", "equation": "LR = AI^2 - 2*SI^2", "role": "feature", "predictor_allowed": True, "notes": "Mentor lambda-rho impedance-form term."},
        {"field": "mr_term", "equation": "MR = SI^2", "role": "feature", "predictor_allowed": True, "notes": "Mentor mu-rho impedance-form term."},
        {"field": "brittleness_youngs_pct", "equation": "BR_YM = ((YM_Mpsi - 1)/(8 - 1))*100", "role": "feature", "predictor_allowed": True, "notes": "Rickman-style brittleness from Young's modulus; clipped to 0-100 by default."},
        {"field": "brittleness_poisson_pct", "equation": "BR_PR = ((PR - 0.40)/(0.15 - 0.40))*100", "role": "feature", "predictor_allowed": True, "notes": "Rickman-style brittleness from Poisson's ratio; clipped to 0-100 by default."},
        {"field": "brittleness_total_pct", "equation": "BRIT_total = (BR_YM + BR_PR)/2", "role": "feature", "predictor_allowed": True, "notes": "Total brittleness feature from mentor geomechanics set."},
    ])

    # Rw estimation and Archie Sw/Sh baseline.
    rw_by_well = estimate_rw_by_well(out)
    out["rw_est_ohm_m"] = out["well_alias"].map(rw_by_well)
    out["sw_archie_calc"] = np.nan
    out["sh_archie_calc"] = np.nan
    for alias, meta in WELL_METADATA.items():
        idx = out["well_alias"] == alias
        a, m, n = meta.get("archie_a", np.nan), meta.get("archie_m", np.nan), meta.get("archie_n", np.nan)
        rw = rw_by_well.get(alias, np.nan)
        if np.isfinite(a) and np.isfinite(m) and np.isfinite(n) and np.isfinite(rw):
            phi_i = out.loc[idx, "phi_effective_for_equations"].clip(0.001, 0.7)
            rt_i = out.loc[idx, "rt_ohm_m"]
            sw = ((a * rw) / (rt_i * (phi_i ** m))) ** (1.0 / n)
            sw = sw.replace([np.inf, -np.inf], np.nan).clip(0.0, 1.0)
            out.loc[idx, "sw_archie_calc"] = sw
            out.loc[idx, "sh_archie_calc"] = (1.0 - sw).clip(0.0, 1.0)
    equations.append({"field": "sw_archie_calc", "equation": "Sw = ((a*Rw)/(Rt*phi^m))^(1/n)", "role": "physics baseline/review", "predictor_allowed": False, "notes": "Rw estimated from S_wr/Swr where available; not a predictor for saturation targets."})
    equations.append({"field": "sh_archie_calc", "equation": "Sh_Archie = 1 - Sw", "role": "physics baseline/review", "predictor_allowed": False, "notes": "Physics baseline/calibration comparison only."})

    # Occurrence screen: rule-based multi-log evidence, not supervised classifier.
    rt_score = ((out["log10_rt"] - np.log10(10)) / (np.log10(100) - np.log10(10))).clip(0.0, 1.0)
    vp_score = (((out["vp_m_s"] / 1000.0) - 2.5) / (4.0 - 2.5)).clip(0.0, 1.0)
    phi_score = ((out["phi_effective_for_equations"] - 0.15) / (0.35 - 0.15)).clip(0.0, 1.0)
    nmr_score = out["sh_nmr_density_calc"].clip(0.0, 1.0)
    archie_score = out["sh_archie_calc"].clip(0.0, 1.0)
    clean_score = out["clean_sand_score"].clip(0.0, 1.0)
    score_components = pd.DataFrame({
        "rt": rt_score,
        "vp": vp_score,
        "phi": phi_score,
        "clean": clean_score,
        "nmr": nmr_score,
        "archie": archie_score,
    })
    weights = pd.Series({"rt": 0.25, "vp": 0.20, "phi": 0.20, "clean": 0.15, "nmr": 0.10, "archie": 0.10})
    weighted = score_components.mul(weights, axis=1)
    available_weight = score_components.notna().mul(weights, axis=1).sum(axis=1)
    out["occurrence_probability_screen"] = (weighted.sum(axis=1) / available_weight).clip(0.0, 1.0)
    out.loc[available_weight < 0.35, "occurrence_probability_screen"] = np.nan

    def class_from_row(row):
        s = row["occurrence_probability_screen"]
        if pd.isna(s):
            return "insufficient_evidence"
        if s < 0.25:
            return "no_hydrate_or_water_like"
        if s < 0.45:
            return "possible_hydrate"
        if row.get("clean_sand_score", np.nan) < 0.35:
            return "ambiguous_resistive_or_stiff_anomaly"
        return "probable_pore_filling_hydrate"

    out["hydrate_occurrence_screen"] = out.apply(class_from_row, axis=1)
    equations.append({"field": "hydrate_occurrence_screen", "equation": "rule score from clean sand + Rt + Vp + phi + NMR/Archie evidence", "role": "rule-based occurrence screen", "predictor_allowed": False, "notes": "Independent log-evidence screen; not used as the supervised occurrence label."})

    # Supervised occurrence label rule: target-side label derived from approved hydrate saturation reference.
    # This is y for the occurrence classifier, never an X predictor.
    sh_ref = out["hydrate_saturation_reference"]
    out["hydrate_occurrence_label"] = np.nan
    out["occurrence_label_status"] = "missing_hydrate_saturation_reference"
    positive = sh_ref >= OCCURRENCE_POSITIVE_SH_THRESHOLD
    negative = sh_ref <= OCCURRENCE_NEGATIVE_SH_THRESHOLD
    gray = sh_ref.notna() & ~(positive | negative)
    out.loc[positive, "hydrate_occurrence_label"] = 1.0
    out.loc[positive, "occurrence_label_status"] = f"positive_Sh_ge_{OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f}"
    out.loc[negative, "hydrate_occurrence_label"] = 0.0
    out.loc[negative, "occurrence_label_status"] = f"negative_Sh_le_{OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f}"
    out.loc[gray, "occurrence_label_status"] = f"gray_zone_{OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f}_to_{OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f}_omitted"
    equations.append({"field": "hydrate_occurrence_label", "equation": f"label=1 if Sh >= {OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f}; label=0 if Sh <= {OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f}; gray zone omitted", "role": "classification target", "predictor_allowed": False, "notes": "Mentor-review occurrence label rule derived from approved S_h/Sgh/Sh target only; blocked from X_allowed."})

    # Cleanup obvious impossible elastic values before QC.
    for col in ["vp_vs_ratio", "poisson_ratio", "bulk_modulus_gpa", "shear_modulus_gpa", "youngs_modulus_gpa", "youngs_modulus_mpsi", "lambda_rho", "mu_rho", "lr_term", "mr_term", "brittleness_youngs_pct", "brittleness_poisson_pct", "brittleness_total_pct"]:
        out.loc[~np.isfinite(out[col]), col] = np.nan

    # QC and caveat flags. In normalized-input mode, these are review flags, not hard exclusions.
    raw_required = ["gr_api", "rhob_g_cc", "density_porosity_vv", "rt_ohm_m", "vp_m_s", "vs_m_s"]
    out["qc_required_log_coverage"] = out[[c for c in raw_required if c in out.columns]].notna().mean(axis=1)
    out["qc_low_required_log_coverage_flag"] = out["qc_required_log_coverage"] < 0.60
    out["qc_missing_rt_flag"] = out["rt_ohm_m"].isna()
    out["qc_missing_velocity_flag"] = out[["vp_m_s", "vs_m_s"]].isna().any(axis=1)
    out["qc_missing_porosity_flag"] = out[["density_porosity_vv", "phi_density_calc", "phi_effective_for_equations"]].isna().all(axis=1)

    cal = out["caliper_raw"]
    cal_med = cal.dropna().median() if cal.notna().any() else np.nan
    out["qc_caliper_status"] = "missing_caliper"
    if pd.notna(cal_med) and 2.0 <= cal_med <= 20.0:
        out["qc_caliper_status"] = np.where(cal.between(CALIPER_EXPECTED_MIN, CALIPER_EXPECTED_MAX), "physical_range_pass", "physical_range_review")
    elif pd.notna(cal_med):
        out["qc_caliper_status"] = "normalized_context_only"
    out["qc_bad_caliper_flag"] = out["qc_caliper_status"].eq("physical_range_review")

    out["qc_elastic_invalid_flag"] = (
        (out["vp_vs_ratio"] <= 1.0) |
        (out["poisson_ratio"] < -0.2) |
        (out["poisson_ratio"] > 0.5) |
        (out["shear_modulus_gpa"] <= 0) |
        (out["bulk_modulus_gpa"] <= 0)
    ).fillna(True)
    out["qc_normalized_input_mode"] = bool(NORMALIZED_INPUT_MODE)
    out["qc_deep_resistivity_alias_confirmed"] = bool(DEEP_RESISTIVITY_ALIAS_CONFIRMED)
    out["qc_status"] = np.where(
        out[["qc_low_required_log_coverage_flag", "qc_missing_rt_flag", "qc_missing_velocity_flag", "qc_missing_porosity_flag", "qc_bad_caliper_flag"]].any(axis=1),
        "review",
        "pass"
    )
    equations.append({"field": "qc_status", "equation": "review if low coverage, missing Rt/velocity/porosity, or physical caliper outside expected range", "role": "QC/caveat", "predictor_allowed": False, "notes": "All QC flags are exported for mentor review; normalized-input mode prevents physical overclaiming."})
    equations.append({"field": "normalized_input_mode", "equation": "all non-depth workbook curves treated as normalized inputs", "role": "runtime caveat", "predictor_allowed": "n/a", "notes": "Elastic and Archie outputs are first-run proxy/calibration features until unnormalized physical-unit logs are supplied."})

    eq_df = pd.DataFrame(equations)
    return out, eq_df

features_df, equations_used = add_equation_features(standardized)
print("Equation features added:", len(features_df.columns))
print("Rw estimates:")
display(features_df[["well_alias", "well_name", "rw_est_ohm_m"]].drop_duplicates())
print("Occurrence screen counts:")
display(features_df["hydrate_occurrence_screen"].value_counts(dropna=False).to_frame("count"))
print("Occurrence label counts from S_h rule:")
display(features_df[["occurrence_label_status", "hydrate_occurrence_label"]].value_counts(dropna=False).to_frame("rows"))
print("QC status counts:")
display(features_df["qc_status"].value_counts(dropna=False).to_frame("rows"))
print("Density-porosity source counts:")
display(features_df.groupby(["well_alias", "phi_effective_source"], dropna=False).size().reset_index(name="rows"))


## 3. Train ML models under single-well transfer validation

In [ ]:
# =============================================================================
# ML models — V9 geomechanics + single-well transfer workflow
# =============================================================================

# V10 keeps the V9 validation/modeling design and adds explicit source-policy audits:
# 1) train on one calibration well only (default WellC / Mount Elbert),
# 2) select model/feature set using only that training well via depth-block CV,
# 3) evaluate on the remaining wells as external transfer wells,
# 4) report combined transfer metrics and per-transfer-well metrics,
# 5) keep normalized-input protection and target-leakage blocking.

MEASURED_FEATURES = [
    "gr_api",
    "rhob_g_cc",
    "density_porosity_vv",
    "neutron_porosity_vv",
    "rt_ohm_m",
    "vp_m_s",
    "vs_m_s",
]
SAFE_TRANSFORM_FEATURES = [
    "log10_rt",
    "clean_sand_score",
]
PROXY_EQUATION_FEATURES = [
    "vp_vs_ratio",
    "acoustic_impedance",
    "shear_impedance",
    "shear_modulus_gpa",
    "bulk_modulus_gpa",
    "youngs_modulus_gpa",
    "youngs_modulus_mpsi",
    "poisson_ratio",
    "lambda_rho",
    "mu_rho",
    "lr_term",
    "mr_term",
    "brittleness_youngs_pct",
    "brittleness_poisson_pct",
    "brittleness_total_pct",
    "phi_density_calc",
    "phi_effective_for_equations",
    "vsh_larionov_tertiary",
]
CONTEXT_FEATURES = [
    "pressure_hydrostatic_mpa",
]

if INCLUDE_NMR_POROSITY_AS_FEATURE:
    MEASURED_FEATURES.append("nmr_porosity_vv")

# Direct target-derived fields and review outputs are deliberately excluded from X_allowed.
BLOCKED_FEATURES = {
    "hydrate_saturation_reference",
    "water_saturation_reference",
    "sh_nmr_density_calc",
    "sw_archie_calc",
    "sh_archie_calc",
    "occurrence_probability_screen",
    "hydrate_occurrence_screen",
    "hydrate_occurrence_label",
    "occurrence_label_status",
    "qc_status",
    "well_alias",
    "well_name",
    "site",
    "source_workbook",
    "source_sheet",
    "source_layout",
    "source_header_contract",
    "source_hydrate_header",
    "source_water_header",
    "phi_effective_source",
    "lat",
    "lon",
}

REGRESSION_MODELS = {
    "mean_baseline": DummyRegressor(strategy="mean"),
    "ridge_a1": Ridge(alpha=1.0, random_state=RANDOM_SEED),
    "ridge_a10": Ridge(alpha=10.0, random_state=RANDOM_SEED),
    "random_forest_shallow": RandomForestRegressor(n_estimators=80, max_depth=4, min_samples_leaf=5, random_state=RANDOM_SEED, n_jobs=1),
    "gradient_boosting_shallow": GradientBoostingRegressor(n_estimators=90, learning_rate=0.04, max_depth=2, random_state=RANDOM_SEED),
    "gradient_boosting_huber": GradientBoostingRegressor(n_estimators=100, learning_rate=0.04, max_depth=2, loss="huber", random_state=RANDOM_SEED),
}

CLASSIFICATION_MODELS = {
    "majority_baseline": DummyClassifier(strategy="most_frequent"),
    "logistic_balanced": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED),
    "random_forest_cls": RandomForestClassifier(n_estimators=90, max_depth=4, min_samples_leaf=4, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=1),
    "gradient_boosting_cls": GradientBoostingClassifier(n_estimators=90, learning_rate=0.04, max_depth=2, random_state=RANDOM_SEED),
}


def unique_preserve_order(values: list[str]) -> list[str]:
    out = []
    seen = set()
    for v in values:
        if v not in seen:
            out.append(v)
            seen.add(v)
    return out


def wells_label(wells: list[str] | str) -> str:
    if isinstance(wells, str):
        return wells
    return "+".join(wells)


def candidate_feature_sets() -> dict[str, list[str]]:
    feature_sets = {
        "measured_only": MEASURED_FEATURES,
        "safe_normalized": MEASURED_FEATURES + SAFE_TRANSFORM_FEATURES,
    }
    if EVALUATE_PROXY_FEATURE_SET_FOR_REVIEW:
        proxy_cols = MEASURED_FEATURES + SAFE_TRANSFORM_FEATURES + PROXY_EQUATION_FEATURES
        if ALLOW_PRESSURE_CONTEXT_IN_PRIMARY_MODEL:
            proxy_cols = proxy_cols + CONTEXT_FEATURES
        feature_sets["equation_proxy_review"] = proxy_cols
    if ALLOW_PRESSURE_CONTEXT_IN_PRIMARY_MODEL:
        feature_sets["safe_plus_pressure_context"] = MEASURED_FEATURES + SAFE_TRANSFORM_FEATURES + CONTEXT_FEATURES
    return {k: unique_preserve_order(v) for k, v in feature_sets.items()}


PRIMARY_ALLOWED_FEATURE_SETS = {"measured_only", "safe_normalized"}
if ALLOW_PROXY_EQUATION_FEATURES_IN_PRIMARY_MODEL:
    PRIMARY_ALLOWED_FEATURE_SETS.add("equation_proxy_review")
if ALLOW_PRESSURE_CONTEXT_IN_PRIMARY_MODEL:
    PRIMARY_ALLOWED_FEATURE_SETS.add("safe_plus_pressure_context")


def available_feature_sets(train_df: pd.DataFrame) -> tuple[dict[str, list[str]], pd.DataFrame]:
    rows = []
    out: dict[str, list[str]] = {}
    for set_name, cols in candidate_feature_sets().items():
        candidates = [c for c in cols if c in train_df.columns and c not in BLOCKED_FEATURES]
        if not candidates:
            out[set_name] = []
            rows.append({"feature_set": set_name, "feature": "none", "coverage": np.nan, "selected": False, "primary_eligible": set_name in PRIMARY_ALLOWED_FEATURE_SETS})
            continue
        coverage = train_df[candidates].notna().mean().sort_values(ascending=False)
        selected = coverage[coverage >= MIN_TRAIN_FEATURE_COVERAGE].index.tolist()
        out[set_name] = selected
        for feature, cov in coverage.items():
            rows.append({
                "feature_set": set_name,
                "feature": feature,
                "coverage": float(cov),
                "selected": feature in selected,
                "primary_eligible": set_name in PRIMARY_ALLOWED_FEATURE_SETS,
                "normalized_input_mode": NORMALIZED_INPUT_MODE,
            })
    return out, pd.DataFrame(rows)


def row_completeness_mask(df: pd.DataFrame, feature_cols: list[str]) -> pd.Series:
    if not feature_cols:
        return pd.Series(False, index=df.index)
    return df[feature_cols].notna().mean(axis=1) >= MIN_ROW_FEATURE_FRACTION


def target_rows(df: pd.DataFrame, target_col: str, wells: list[str], feature_cols: list[str] | None = None) -> pd.DataFrame:
    sub = df[df["well_alias"].isin(wells)].copy()
    sub = sub[sub[target_col].notna()].copy()
    if feature_cols:
        sub = sub[row_completeness_mask(sub, feature_cols)].copy()
    return sub


def equal_well_weights(df: pd.DataFrame) -> np.ndarray:
    counts = df["well_alias"].value_counts().to_dict()
    w = df["well_alias"].map(lambda x: 1.0 / max(counts.get(x, 1), 1)).astype(float).to_numpy()
    return w / np.nanmean(w)


def regression_sample_weights(df: pd.DataFrame, target_col: str) -> np.ndarray:
    w = equal_well_weights(df)
    if APPLY_TARGET_BIN_WEIGHTS and target_col in df.columns:
        y = pd.to_numeric(df[target_col], errors="coerce").clip(0, 1)
        bins = pd.cut(y, bins=SATURATION_BIN_EDGES, labels=False, include_lowest=True)
        counts = bins.value_counts(dropna=True).to_dict()
        used_bins = max(len(counts), 1)
        bweights = bins.map(lambda b: len(y) / (used_bins * counts.get(b, 1)) if pd.notna(b) else 1.0).astype(float).to_numpy()
        bweights = np.clip(bweights, 0.35, 4.0)
        w = w * bweights
    return w / np.nanmean(w)


def classification_sample_weights(df: pd.DataFrame, target_col: str) -> np.ndarray:
    w = equal_well_weights(df)
    y = df[target_col].astype(int)
    counts = y.value_counts().to_dict()
    class_w = y.map(lambda c: len(y) / (2.0 * max(counts.get(c, 1), 1))).astype(float).to_numpy()
    w = w * class_w
    return w / np.nanmean(w)


def fit_pipeline(model, X, y, sample_weight=None) -> Pipeline:
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler()),
        ("model", clone(model)),
    ])
    if sample_weight is not None:
        try:
            pipe.fit(X, y, model__sample_weight=sample_weight)
        except Exception:
            pipe.fit(X, y)
    else:
        pipe.fit(X, y)
    return pipe


def metric_dict(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "n": int(np.isfinite(y_true).sum()),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(math.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
        "bias": float(np.nanmean(y_pred - y_true)),
        "prediction_min": float(np.nanmin(y_pred)),
        "prediction_max": float(np.nanmax(y_pred)),
    }


def classifier_probability(pipe: Pipeline, X: pd.DataFrame) -> np.ndarray:
    model = pipe.named_steps["model"]
    if hasattr(pipe, "predict_proba"):
        proba = pipe.predict_proba(X)
        classes = list(getattr(model, "classes_", []))
        if 1 in classes:
            return proba[:, classes.index(1)]
        return proba[:, -1]
    pred = pipe.predict(X)
    return np.asarray(pred, dtype=float)


def classification_metric_dict(y_true, proba, pred_label) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=int)
    proba = np.asarray(proba, dtype=float)
    pred_label = np.asarray(pred_label, dtype=int)
    out = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, pred_label)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred_label)),
        "precision": float(precision_score(y_true, pred_label, zero_division=0)),
        "recall": float(recall_score(y_true, pred_label, zero_division=0)),
        "f1": float(f1_score(y_true, pred_label, zero_division=0)),
        "brier": float(brier_score_loss(y_true, np.clip(proba, 0, 1))) if len(np.unique(y_true)) == 2 else np.nan,
        "positive_rate_reference": float(np.mean(y_true == 1)),
        "positive_rate_predicted": float(np.mean(pred_label == 1)),
        "probability_min": float(np.nanmin(proba)),
        "probability_max": float(np.nanmax(proba)),
    }
    out["roc_auc"] = float(roc_auc_score(y_true, proba)) if len(np.unique(y_true)) == 2 else np.nan
    return out


def make_depth_block_groups(train: pd.DataFrame, n_splits: int = DEPTH_BLOCK_CV_N_SPLITS) -> tuple[np.ndarray, str, int]:
    """Create CV groups without using external transfer wells.

    If multiple wells are in the train set, use well groups. If one well is in the
    train set, split the training well into contiguous depth blocks.
    """
    well_groups = train["well_alias"].astype(str).to_numpy()
    unique_wells = np.unique(well_groups)
    if len(unique_wells) >= 2:
        return well_groups, "well_group_cv_inside_train", len(unique_wells)

    depth = pd.to_numeric(train.get("depth_m"), errors="coerce")
    if depth.notna().sum() < 20:
        # Fallback to row-order blocks if depth is missing, still using only training rows.
        order = pd.Series(np.arange(len(train)), index=train.index)
    else:
        order = depth.rank(method="first")
    q = min(int(n_splits), max(2, len(train) // 40))
    q = max(2, q)
    try:
        groups = pd.qcut(order, q=q, labels=False, duplicates="drop")
    except Exception:
        groups = pd.Series(np.arange(len(train)) % q, index=train.index)
    groups = pd.Series(groups, index=train.index).astype("Int64").fillna(0).astype(int).to_numpy()
    n_groups = len(np.unique(groups))
    return groups, "single_train_well_depth_block_cv", n_groups


def saturation_bin_metrics(pred_df: pd.DataFrame, target: str, model: str, feature_set: str, validation_scope: str = "combined_transfer") -> pd.DataFrame:
    if pred_df.empty:
        return pd.DataFrame()
    tmp = pred_df.copy()
    tmp["reference_bin"] = pd.cut(tmp["reference"].clip(0, 1), bins=SATURATION_BIN_EDGES, labels=SATURATION_BIN_LABELS, include_lowest=True)
    rows = []
    for bin_name, sub in tmp.groupby("reference_bin", dropna=False):
        if len(sub) == 0:
            continue
        m = metric_dict(sub["reference"], sub["prediction"])
        m.update({
            "target": target,
            "model": model,
            "feature_set": feature_set,
            "validation_scope": validation_scope,
            "reference_bin": str(bin_name),
            "reference_mean": float(sub["reference"].mean()),
            "prediction_mean": float(sub["prediction"].mean()),
        })
        rows.append(m)
    tmp2 = tmp[tmp["reference"].notna()].copy()
    if len(tmp2):
        y_true = (tmp2["reference"] >= OCCURRENCE_POSITIVE_SH_THRESHOLD).astype(int)
        y_pred = (tmp2["prediction"] >= OCCURRENCE_POSITIVE_SH_THRESHOLD).astype(int)
        rows.append({
            "target": target,
            "model": model,
            "feature_set": feature_set,
            "validation_scope": validation_scope,
            "reference_bin": f"threshold_Sh_ge_{OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f}",
            "n": int(len(tmp2)),
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "bias": np.nan,
            "prediction_min": np.nan,
            "prediction_max": np.nan,
            "reference_mean": float(y_true.mean()),
            "prediction_mean": float(y_pred.mean()),
            "occurrence_accuracy_from_regression": float(accuracy_score(y_true, y_pred)),
            "occurrence_f1_from_regression": float(f1_score(y_true, y_pred, zero_division=0)),
            "occurrence_recall_from_regression": float(recall_score(y_true, y_pred, zero_division=0)),
        })
    return pd.DataFrame(rows)


def evaluate_regression_model(pipe: Pipeline, valid: pd.DataFrame, features: list[str], target_col: str) -> tuple[dict[str, float], pd.DataFrame]:
    X_valid = valid[features]
    y_valid = valid[target_col].astype(float)
    pred_raw = pipe.predict(X_valid)
    pred = np.clip(pred_raw, 0.0, 1.0)
    m = metric_dict(y_valid, pred)
    tmp_cols = [
        "well_alias", "well_name", "site", "depth_m", "depth_ft", target_col,
        "hydrate_occurrence_screen", "occurrence_probability_screen", "hydrate_occurrence_label",
        "occurrence_label_status", "qc_status", "sh_nmr_density_calc", "sh_archie_calc",
        "sw_archie_calc", "rw_est_ohm_m",
    ]
    tmp_cols = [c for c in tmp_cols if c in valid.columns]
    pred_df = valid[tmp_cols].copy().rename(columns={target_col: "reference"})
    pred_df["prediction"] = pred
    pred_df["prediction_raw"] = pred_raw
    pred_df["residual"] = pred_df["prediction"] - pred_df["reference"]
    return m, pred_df


def per_well_regression_metrics(pred_df: pd.DataFrame, base: dict[str, Any]) -> pd.DataFrame:
    rows = []
    if pred_df.empty:
        return pd.DataFrame(rows)
    for well, sub in pred_df.groupby("well_alias", dropna=False):
        if len(sub) < 2:
            continue
        m = metric_dict(sub["reference"], sub["prediction"])
        row = dict(base)
        row.update(m)
        row["validation_scope"] = "per_transfer_well"
        row["validation_well"] = well
        row["validation_rows"] = len(sub)
        rows.append(row)
    return pd.DataFrame(rows)


def group_cv_regression(df: pd.DataFrame, target_name: str, target_col: str, train_wells: list[str]) -> tuple[pd.DataFrame, dict[str, list[str]], pd.DataFrame]:
    train0 = df[df["well_alias"].isin(train_wells) & df[target_col].notna()].copy()
    feature_sets, feature_catalog = available_feature_sets(train0)
    rows = []
    for feature_set_name, features in feature_sets.items():
        if not features:
            continue
        train = target_rows(df, target_col, train_wells, features)
        groups, cv_mode, n_groups = make_depth_block_groups(train)
        for model_name, model in REGRESSION_MODELS.items():
            fold_rows = []
            if len(train) < 20 or n_groups < 2:
                rows.append({
                    "target": target_name,
                    "task_type": "regression",
                    "feature_set": feature_set_name,
                    "model": model_name,
                    "cv_status": "not_enough_groups_or_rows",
                    "cv_mode": cv_mode,
                    "cv_folds": 0,
                    "cv_rmse_mean": np.nan,
                    "cv_mae_mean": np.nan,
                    "cv_r2_mean": np.nan,
                    "primary_eligible": feature_set_name in PRIMARY_ALLOWED_FEATURE_SETS,
                })
                continue
            cv = GroupKFold(n_splits=n_groups)
            for fold_i, (tr_idx, va_idx) in enumerate(cv.split(train[features], train[target_col], groups=groups), start=1):
                tr = train.iloc[tr_idx].copy()
                va = train.iloc[va_idx].copy()
                if len(tr) < 10 or len(va) < 5:
                    continue
                weights = regression_sample_weights(tr, target_col)
                try:
                    pipe = fit_pipeline(model, tr[features], tr[target_col].astype(float), sample_weight=weights)
                    m, _ = evaluate_regression_model(pipe, va, features, target_col)
                    m.update({"fold": fold_i, "heldout_train_block": str(np.unique(groups[va_idx]).tolist()), "cv_mode": cv_mode})
                    fold_rows.append(m)
                except Exception as exc:
                    fold_rows.append({"fold": fold_i, "rmse": np.nan, "mae": np.nan, "r2": np.nan, "cv_mode": cv_mode, "note": str(exc)})
            fold_df = pd.DataFrame(fold_rows)
            rows.append({
                "target": target_name,
                "task_type": "regression",
                "feature_set": feature_set_name,
                "model": model_name,
                "cv_status": "ok" if fold_df.get("rmse", pd.Series(dtype=float)).notna().any() else "failed",
                "cv_mode": cv_mode,
                "cv_folds": int(fold_df.get("rmse", pd.Series(dtype=float)).notna().sum()),
                "cv_rmse_mean": float(fold_df["rmse"].mean()) if "rmse" in fold_df else np.nan,
                "cv_rmse_std": float(fold_df["rmse"].std()) if "rmse" in fold_df else np.nan,
                "cv_mae_mean": float(fold_df["mae"].mean()) if "mae" in fold_df else np.nan,
                "cv_r2_mean": float(fold_df["r2"].mean()) if "r2" in fold_df else np.nan,
                "primary_eligible": feature_set_name in PRIMARY_ALLOWED_FEATURE_SETS,
                "feature_count": len(features),
                "train_rows_cv_pool": len(train),
                "train_wells": wells_label(train_wells),
            })
    return pd.DataFrame(rows), feature_sets, feature_catalog


def select_regression_candidate(cv_df: pd.DataFrame) -> pd.Series:
    eligible = cv_df[
        cv_df["primary_eligible"].fillna(False)
        & cv_df["cv_rmse_mean"].notna()
        & ~cv_df["model"].str.contains("baseline", case=False, na=False)
    ].copy()
    if eligible.empty:
        eligible = cv_df[cv_df["cv_rmse_mean"].notna()].copy()
    if eligible.empty:
        raise ValueError("No regression candidate produced valid training-only CV metrics")
    return eligible.sort_values(["cv_rmse_mean", "cv_mae_mean", "feature_count"], ascending=[True, True, True]).iloc[0]


def run_regression_target(df: pd.DataFrame, target_name: str, target_col: str, train_wells: list[str], validation_wells: list[str], diagnostic_only: bool = False) -> dict[str, Any]:
    cv_df, feature_sets, feature_catalog = group_cv_regression(df, target_name, target_col, train_wells)
    if diagnostic_only:
        valid_cv = cv_df[cv_df["cv_rmse_mean"].notna()].copy()
        if valid_cv.empty:
            selected = pd.Series({"feature_set": "safe_normalized", "model": "ridge_a1", "cv_rmse_mean": np.nan, "cv_mae_mean": np.nan, "cv_r2_mean": np.nan, "primary_eligible": True})
        else:
            selected = select_regression_candidate(cv_df)
    else:
        selected = select_regression_candidate(cv_df)

    selected_feature_set = str(selected["feature_set"])
    selected_model_name = str(selected["model"])
    features = feature_sets.get(selected_feature_set) or available_feature_sets(df[df["well_alias"].isin(train_wells)].copy())[0].get(selected_feature_set, [])
    if not features:
        raise ValueError(f"No selected features for {target_name}/{selected_feature_set}")

    train = target_rows(df, target_col, train_wells, features)
    valid = target_rows(df, target_col, validation_wells, features)
    if len(train) < 20:
        raise ValueError(f"Not enough training rows for {target_name}: {len(train)}")
    if len(valid) < 5:
        raise ValueError(f"Not enough validation rows for {target_name} on {wells_label(validation_wells)}: {len(valid)}")

    selected_model = REGRESSION_MODELS[selected_model_name]
    selected_pipe = fit_pipeline(selected_model, train[features], train[target_col].astype(float), sample_weight=regression_sample_weights(train, target_col))
    transfer_m, transfer_pred = evaluate_regression_model(selected_pipe, valid, features, target_col)
    base = {
        "target": target_name,
        "target_column": target_col,
        "task_type": "regression",
        "model": selected_model_name,
        "feature_set": selected_feature_set,
        "selection_stage": "external_transfer_after_training_well_depth_block_cv" if not diagnostic_only else "diagnostic_external_transfer_after_training_well_cv",
        "train_wells": wells_label(train_wells),
        "validation_well": wells_label(validation_wells),
        "validation_wells": wells_label(validation_wells),
        "validation_scope": "combined_transfer_wells",
        "feature_count": len(features),
        "train_rows": len(train),
        "validation_rows": len(valid),
        "cv_rmse_mean_for_selection": selected.get("cv_rmse_mean", np.nan),
        "cv_mae_mean_for_selection": selected.get("cv_mae_mean", np.nan),
        "cv_mode_for_selection": selected.get("cv_mode", TRAINING_SELECTION_MODE),
        "diagnostic_only": bool(diagnostic_only),
    }
    transfer_m.update(base)
    transfer_pred["target"] = target_name
    transfer_pred["model"] = selected_model_name
    transfer_pred["feature_set"] = selected_feature_set
    transfer_pred["validation_scope"] = "combined_transfer_wells"

    per_well_metrics = per_well_regression_metrics(transfer_pred, base)

    # Evaluate all candidates on transfer wells for transparency, but do not use transfer metrics for selection.
    transfer_rows = [transfer_m]
    all_candidate_predictions = [transfer_pred]
    for feature_set_name, feature_cols in feature_sets.items():
        if not feature_cols:
            continue
        tr = target_rows(df, target_col, train_wells, feature_cols)
        va = target_rows(df, target_col, validation_wells, feature_cols)
        if len(tr) < 20 or len(va) < 5:
            continue
        for model_name, model in REGRESSION_MODELS.items():
            if model_name == selected_model_name and feature_set_name == selected_feature_set:
                continue
            try:
                pipe = fit_pipeline(model, tr[feature_cols], tr[target_col].astype(float), sample_weight=regression_sample_weights(tr, target_col))
                m, p = evaluate_regression_model(pipe, va, feature_cols, target_col)
                m.update({
                    "target": target_name,
                    "target_column": target_col,
                    "task_type": "regression",
                    "model": model_name,
                    "feature_set": feature_set_name,
                    "selection_stage": "external_transfer_transparency_only_not_used_for_selection",
                    "train_wells": wells_label(train_wells),
                    "validation_well": wells_label(validation_wells),
                    "validation_wells": wells_label(validation_wells),
                    "validation_scope": "combined_transfer_wells",
                    "feature_count": len(feature_cols),
                    "train_rows": len(tr),
                    "validation_rows": len(va),
                    "cv_rmse_mean_for_selection": cv_df.loc[(cv_df["model"] == model_name) & (cv_df["feature_set"] == feature_set_name), "cv_rmse_mean"].min() if len(cv_df) else np.nan,
                    "cv_mae_mean_for_selection": cv_df.loc[(cv_df["model"] == model_name) & (cv_df["feature_set"] == feature_set_name), "cv_mae_mean"].min() if len(cv_df) else np.nan,
                    "cv_mode_for_selection": TRAINING_SELECTION_MODE,
                    "diagnostic_only": bool(diagnostic_only),
                })
                p["target"] = target_name
                p["model"] = model_name
                p["feature_set"] = feature_set_name
                p["validation_scope"] = "combined_transfer_wells"
                transfer_rows.append(m)
                all_candidate_predictions.append(p)
            except Exception:
                continue

    transfer_metrics = pd.DataFrame(transfer_rows).sort_values(["selection_stage", "rmse", "mae"]).reset_index(drop=True)
    all_preds = pd.concat(all_candidate_predictions, ignore_index=True)
    bin_metrics = saturation_bin_metrics(transfer_pred, target_name, selected_model_name, selected_feature_set)

    # Final refit remains on the single calibration well for the deployed transfer model.
    final_train = train.copy()
    final_pipe = fit_pipeline(selected_model, final_train[features], final_train[target_col].astype(float), sample_weight=regression_sample_weights(final_train, target_col))

    try:
        perm = permutation_importance(selected_pipe, valid[features], valid[target_col].astype(float), n_repeats=8, random_state=RANDOM_SEED, scoring="neg_root_mean_squared_error")
        imp = pd.DataFrame({
            "target": target_name,
            "model": selected_model_name,
            "feature_set": selected_feature_set,
            "feature": features,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
            "importance_scoring": "neg_root_mean_squared_error",
            "importance_caveat": "computed on external transfer wells for review; not used for model selection",
        }).sort_values("importance_mean", ascending=False)
    except Exception as exc:
        imp = pd.DataFrame({"target": [target_name], "model": [selected_model_name], "feature_set": [selected_feature_set], "feature": ["importance_failed"], "importance_mean": [np.nan], "importance_std": [np.nan], "note": [str(exc)]})

    return {
        "task_type": "regression",
        "target": target_name,
        "target_col": target_col,
        "train_wells": train_wells,
        "validation_wells": validation_wells,
        "validation_well": wells_label(validation_wells),
        "features": features,
        "feature_set": selected_feature_set,
        "cv_metrics": cv_df,
        "feature_catalog": feature_catalog,
        "metrics": transfer_metrics,
        "selected_blind_metrics": pd.DataFrame([transfer_m]),
        "transfer_metrics_by_well": per_well_metrics,
        "predictions": all_preds,
        "selected_predictions": transfer_pred,
        "bin_metrics": bin_metrics,
        "feature_importance": imp,
        "selected_model_name": selected_model_name,
        "selected_validation_pipeline": selected_pipe,
        "final_pipeline": final_pipe,
        "final_train_rows": len(final_train),
        "final_train_wells": train_wells,
        "diagnostic_only": bool(diagnostic_only),
    }


def group_cv_classification(df: pd.DataFrame, target_col: str, train_wells: list[str]) -> tuple[pd.DataFrame, dict[str, list[str]], pd.DataFrame]:
    train0 = df[df["well_alias"].isin(train_wells) & df[target_col].notna()].copy()
    feature_sets, feature_catalog = available_feature_sets(train0)
    rows = []
    for feature_set_name, features in feature_sets.items():
        if not features:
            continue
        train = target_rows(df, target_col, train_wells, features)
        groups, cv_mode, n_groups = make_depth_block_groups(train)
        for model_name, model in CLASSIFICATION_MODELS.items():
            fold_rows = []
            if len(train) < 20 or n_groups < 2 or train[target_col].nunique(dropna=True) < 2:
                rows.append({
                    "target": "hydrate_occurrence",
                    "task_type": "classification",
                    "feature_set": feature_set_name,
                    "model": model_name,
                    "cv_status": "not_enough_groups_rows_or_classes",
                    "cv_mode": cv_mode,
                    "cv_folds": 0,
                    "cv_f1_mean": np.nan,
                    "cv_balanced_accuracy_mean": np.nan,
                    "cv_roc_auc_mean": np.nan,
                    "primary_eligible": feature_set_name in PRIMARY_ALLOWED_FEATURE_SETS,
                    "train_wells": wells_label(train_wells),
                })
                continue
            cv = GroupKFold(n_splits=n_groups)
            for fold_i, (tr_idx, va_idx) in enumerate(cv.split(train[features], train[target_col], groups=groups), start=1):
                tr = train.iloc[tr_idx].copy()
                va = train.iloc[va_idx].copy()
                if len(tr) < 10 or len(va) < 5 or tr[target_col].nunique(dropna=True) < 2:
                    continue
                try:
                    pipe = fit_pipeline(model, tr[features], tr[target_col].astype(int), sample_weight=classification_sample_weights(tr, target_col))
                    proba = classifier_probability(pipe, va[features])
                    pred_label = (proba >= OCCURRENCE_CLASSIFICATION_THRESHOLD).astype(int)
                    m = classification_metric_dict(va[target_col].astype(int), proba, pred_label)
                    m.update({"fold": fold_i, "heldout_train_block": str(np.unique(groups[va_idx]).tolist()), "cv_mode": cv_mode})
                    fold_rows.append(m)
                except Exception as exc:
                    fold_rows.append({"fold": fold_i, "f1": np.nan, "balanced_accuracy": np.nan, "roc_auc": np.nan, "cv_mode": cv_mode, "note": str(exc)})
            fold_df = pd.DataFrame(fold_rows)
            rows.append({
                "target": "hydrate_occurrence",
                "task_type": "classification",
                "feature_set": feature_set_name,
                "model": model_name,
                "cv_status": "ok" if fold_df.get("f1", pd.Series(dtype=float)).notna().any() else "failed",
                "cv_mode": cv_mode,
                "cv_folds": int(fold_df.get("f1", pd.Series(dtype=float)).notna().sum()),
                "cv_f1_mean": float(fold_df["f1"].mean()) if "f1" in fold_df else np.nan,
                "cv_balanced_accuracy_mean": float(fold_df["balanced_accuracy"].mean()) if "balanced_accuracy" in fold_df else np.nan,
                "cv_roc_auc_mean": float(fold_df["roc_auc"].mean()) if "roc_auc" in fold_df else np.nan,
                "primary_eligible": feature_set_name in PRIMARY_ALLOWED_FEATURE_SETS,
                "feature_count": len(features),
                "train_rows_cv_pool": len(train),
                "train_wells": wells_label(train_wells),
            })
    return pd.DataFrame(rows), feature_sets, feature_catalog


def select_classification_candidate(cv_df: pd.DataFrame) -> pd.Series:
    eligible = cv_df[
        cv_df["primary_eligible"].fillna(False)
        & (~cv_df["model"].str.contains("baseline", case=False, na=False))
        & (cv_df[["cv_f1_mean", "cv_balanced_accuracy_mean", "cv_roc_auc_mean"]].notna().any(axis=1))
    ].copy()
    if eligible.empty:
        eligible = cv_df[(~cv_df["model"].str.contains("baseline", case=False, na=False))].copy()
    if eligible.empty:
        eligible = cv_df.copy()
    eligible["selection_score"] = eligible["cv_roc_auc_mean"].fillna(eligible["cv_f1_mean"]).fillna(eligible["cv_balanced_accuracy_mean"])
    if eligible["selection_score"].notna().any():
        return eligible.sort_values(["selection_score", "cv_f1_mean", "cv_balanced_accuracy_mean", "feature_count"], ascending=[False, False, False, True]).iloc[0]
    return pd.Series({"feature_set": "safe_normalized", "model": "logistic_balanced", "selection_score": np.nan, "cv_mode": TRAINING_SELECTION_MODE})


def per_well_classification_metrics(pred_df: pd.DataFrame, base: dict[str, Any]) -> pd.DataFrame:
    rows = []
    if pred_df.empty:
        return pd.DataFrame(rows)
    for well, sub in pred_df.groupby("well_alias", dropna=False):
        if len(sub) < 2:
            continue
        m = classification_metric_dict(sub["reference"].astype(int), sub["occurrence_probability_ml"], sub["predicted_label"].astype(int))
        row = dict(base)
        row.update(m)
        row["validation_scope"] = "per_transfer_well"
        row["validation_well"] = well
        row["validation_rows"] = len(sub)
        rows.append(row)
    return pd.DataFrame(rows)


def run_occurrence_classifier(df: pd.DataFrame, train_wells: list[str], validation_wells: list[str]) -> dict[str, Any]:
    target_name = "hydrate_occurrence"
    target_col = "hydrate_occurrence_label"
    cv_df, feature_sets, feature_catalog = group_cv_classification(df, target_col, train_wells)
    selected = select_classification_candidate(cv_df)
    selected_feature_set = str(selected["feature_set"])
    selected_model_name = str(selected["model"])
    features = feature_sets.get(selected_feature_set) or available_feature_sets(df[df["well_alias"].isin(train_wells)].copy())[0].get(selected_feature_set, [])
    if not features:
        raise ValueError(f"No selected occurrence features for {selected_feature_set}")

    train = target_rows(df, target_col, train_wells, features)
    valid = target_rows(df, target_col, validation_wells, features)
    if len(train) < 20:
        raise ValueError(f"Not enough occurrence training rows: {len(train)}")
    if len(valid) < 5:
        raise ValueError(f"Not enough occurrence validation rows on {wells_label(validation_wells)}: {len(valid)}")
    if train[target_col].nunique(dropna=True) < 2:
        raise ValueError("Occurrence classifier needs both positive and negative labels in training wells")

    model = CLASSIFICATION_MODELS[selected_model_name]
    selected_pipe = fit_pipeline(model, train[features], train[target_col].astype(int), sample_weight=classification_sample_weights(train, target_col))
    proba = classifier_probability(selected_pipe, valid[features])
    pred_label = (proba >= OCCURRENCE_CLASSIFICATION_THRESHOLD).astype(int)
    transfer_m = classification_metric_dict(valid[target_col].astype(int), proba, pred_label)
    base = {
        "target": target_name,
        "target_column": target_col,
        "task_type": "classification",
        "model": selected_model_name,
        "feature_set": selected_feature_set,
        "selection_stage": "external_transfer_after_training_well_depth_block_cv",
        "train_wells": wells_label(train_wells),
        "validation_well": wells_label(validation_wells),
        "validation_wells": wells_label(validation_wells),
        "validation_scope": "combined_transfer_wells",
        "feature_count": len(features),
        "train_rows": len(train),
        "validation_rows": len(valid),
        "positive_threshold_sh": OCCURRENCE_POSITIVE_SH_THRESHOLD,
        "negative_threshold_sh": OCCURRENCE_NEGATIVE_SH_THRESHOLD,
        "classification_threshold": OCCURRENCE_CLASSIFICATION_THRESHOLD,
        "cv_f1_mean_for_selection": selected.get("cv_f1_mean", np.nan),
        "cv_balanced_accuracy_mean_for_selection": selected.get("cv_balanced_accuracy_mean", np.nan),
        "cv_roc_auc_mean_for_selection": selected.get("cv_roc_auc_mean", np.nan),
        "cv_mode_for_selection": selected.get("cv_mode", TRAINING_SELECTION_MODE),
        "diagnostic_only": False,
    }
    transfer_m.update(base)

    tmp_cols = [
        "well_alias", "well_name", "site", "depth_m", "depth_ft", "hydrate_saturation_reference",
        "hydrate_occurrence_label", "occurrence_label_status", "hydrate_occurrence_screen",
        "occurrence_probability_screen", "qc_status", "sh_nmr_density_calc", "sh_archie_calc",
        "sw_archie_calc", "rw_est_ohm_m",
    ]
    tmp_cols = [c for c in tmp_cols if c in valid.columns]
    transfer_pred = valid[tmp_cols].copy()
    transfer_pred["target"] = target_name
    transfer_pred["model"] = selected_model_name
    transfer_pred["feature_set"] = selected_feature_set
    transfer_pred["validation_scope"] = "combined_transfer_wells"
    transfer_pred["reference"] = valid[target_col].astype(int).to_numpy()
    transfer_pred["occurrence_probability_ml"] = np.clip(proba, 0.0, 1.0)
    transfer_pred["predicted_label"] = pred_label
    transfer_pred["label_residual"] = transfer_pred["predicted_label"] - transfer_pred["reference"]

    per_well_metrics = per_well_classification_metrics(transfer_pred, base)

    transfer_rows = [transfer_m]
    all_pred_tables = [transfer_pred]
    for feature_set_name, feature_cols in feature_sets.items():
        if not feature_cols:
            continue
        tr = target_rows(df, target_col, train_wells, feature_cols)
        va = target_rows(df, target_col, validation_wells, feature_cols)
        if len(tr) < 20 or len(va) < 5 or tr[target_col].nunique(dropna=True) < 2:
            continue
        for model_name, mmodel in CLASSIFICATION_MODELS.items():
            if model_name == selected_model_name and feature_set_name == selected_feature_set:
                continue
            try:
                pipe = fit_pipeline(mmodel, tr[feature_cols], tr[target_col].astype(int), sample_weight=classification_sample_weights(tr, target_col))
                p = classifier_probability(pipe, va[feature_cols])
                lab = (p >= OCCURRENCE_CLASSIFICATION_THRESHOLD).astype(int)
                m = classification_metric_dict(va[target_col].astype(int), p, lab)
                m.update({
                    "target": target_name,
                    "target_column": target_col,
                    "task_type": "classification",
                    "model": model_name,
                    "feature_set": feature_set_name,
                    "selection_stage": "external_transfer_transparency_only_not_used_for_selection",
                    "train_wells": wells_label(train_wells),
                    "validation_well": wells_label(validation_wells),
                    "validation_wells": wells_label(validation_wells),
                    "validation_scope": "combined_transfer_wells",
                    "feature_count": len(feature_cols),
                    "train_rows": len(tr),
                    "validation_rows": len(va),
                    "positive_threshold_sh": OCCURRENCE_POSITIVE_SH_THRESHOLD,
                    "negative_threshold_sh": OCCURRENCE_NEGATIVE_SH_THRESHOLD,
                    "classification_threshold": OCCURRENCE_CLASSIFICATION_THRESHOLD,
                    "cv_mode_for_selection": TRAINING_SELECTION_MODE,
                    "diagnostic_only": False,
                })
                tab = va[tmp_cols].copy()
                tab["target"] = target_name
                tab["model"] = model_name
                tab["feature_set"] = feature_set_name
                tab["validation_scope"] = "combined_transfer_wells"
                tab["reference"] = va[target_col].astype(int).to_numpy()
                tab["occurrence_probability_ml"] = np.clip(p, 0.0, 1.0)
                tab["predicted_label"] = lab
                tab["label_residual"] = tab["predicted_label"] - tab["reference"]
                transfer_rows.append(m)
                all_pred_tables.append(tab)
            except Exception:
                continue

    metrics = pd.DataFrame(transfer_rows).sort_values(["selection_stage", "f1", "balanced_accuracy"], ascending=[True, False, False]).reset_index(drop=True)
    predictions = pd.concat(all_pred_tables, ignore_index=True)

    final_train = train.copy()
    final_pipe = fit_pipeline(model, final_train[features], final_train[target_col].astype(int), sample_weight=classification_sample_weights(final_train, target_col))

    try:
        scoring = "roc_auc" if valid[target_col].nunique() == 2 else "accuracy"
        perm = permutation_importance(selected_pipe, valid[features], valid[target_col].astype(int), n_repeats=8, random_state=RANDOM_SEED, scoring=scoring)
        imp = pd.DataFrame({
            "target": target_name,
            "model": selected_model_name,
            "feature_set": selected_feature_set,
            "feature": features,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
            "importance_scoring": scoring,
            "importance_caveat": "computed on external transfer wells for review; not used for model selection",
        }).sort_values("importance_mean", ascending=False)
    except Exception as exc:
        imp = pd.DataFrame({"target": [target_name], "model": [selected_model_name], "feature_set": [selected_feature_set], "feature": ["importance_failed"], "importance_mean": [np.nan], "importance_std": [np.nan], "note": [str(exc)]})

    return {
        "task_type": "classification",
        "target": target_name,
        "target_col": target_col,
        "train_wells": train_wells,
        "validation_wells": validation_wells,
        "validation_well": wells_label(validation_wells),
        "features": features,
        "feature_set": selected_feature_set,
        "cv_metrics": cv_df,
        "feature_catalog": feature_catalog,
        "metrics": metrics,
        "selected_blind_metrics": pd.DataFrame([transfer_m]),
        "transfer_metrics_by_well": per_well_metrics,
        "predictions": predictions,
        "selected_predictions": transfer_pred,
        "feature_importance": imp,
        "selected_model_name": selected_model_name,
        "selected_validation_pipeline": selected_pipe,
        "final_pipeline": final_pipe,
        "final_train_rows": len(final_train),
        "final_train_wells": train_wells,
        "diagnostic_only": False,
    }

# Run single-well transfer outputs.
results = []
results.append(run_regression_target(features_df, "hydrate_saturation", "hydrate_saturation_reference", HYDRATE_TRAIN_WELLS, HYDRATE_VALIDATION_WELLS, diagnostic_only=False))
# Water is retained only as a diagnostic transfer. Wells without water targets are automatically dropped.
results.append(run_regression_target(features_df, "water_saturation", "water_saturation_reference", WATER_TRAIN_WELLS, WATER_VALIDATION_WELLS, diagnostic_only=(WATER_MODEL_MODE == "diagnostic_only")))
occurrence_result = run_occurrence_classifier(features_df, OCCURRENCE_TRAIN_WELLS, OCCURRENCE_VALIDATION_WELLS)

# Consolidated tables.
metrics_df = pd.concat([r["metrics"] for r in results], ignore_index=True)
predictions_df = pd.concat([r["predictions"] for r in results], ignore_index=True)
selected_regression_metrics_df = pd.concat([r["selected_blind_metrics"] for r in results], ignore_index=True)
cv_metrics_df = pd.concat([*[r["cv_metrics"] for r in results], occurrence_result["cv_metrics"]], ignore_index=True)
feature_set_catalog_df = pd.concat([*[r["feature_catalog"] for r in results], occurrence_result["feature_catalog"]], ignore_index=True).drop_duplicates().reset_index(drop=True)
bin_metrics_df = pd.concat([r["bin_metrics"] for r in results if len(r.get("bin_metrics", pd.DataFrame()))], ignore_index=True) if any(len(r.get("bin_metrics", pd.DataFrame())) for r in results) else pd.DataFrame()
occurrence_metrics_df = occurrence_result["metrics"]
occurrence_predictions_df = occurrence_result["predictions"]
feature_importance_df = pd.concat([*[r["feature_importance"] for r in results], occurrence_result["feature_importance"]], ignore_index=True)
transfer_metrics_by_well_df = pd.concat([*[r.get("transfer_metrics_by_well", pd.DataFrame()) for r in results], occurrence_result.get("transfer_metrics_by_well", pd.DataFrame())], ignore_index=True, sort=False)

selected_rows = []
for r in results:
    best = r["selected_blind_metrics"].iloc[0].to_dict()
    selected_rows.append({
        "target": r["target"],
        "task_type": r["task_type"],
        "selected_model": r["selected_model_name"],
        "feature_set": r["feature_set"],
        "validation_well": r["validation_well"],
        "validation_wells": r["validation_well"],
        "train_wells": "+".join(r["train_wells"]),
        "final_train_wells": "+".join(r["final_train_wells"]),
        "features_used": ", ".join(r["features"]),
        "feature_count": len(r["features"]),
        "final_train_rows": r["final_train_rows"],
        "validation_rmse": best.get("rmse", np.nan),
        "validation_mae": best.get("mae", np.nan),
        "validation_r2": best.get("r2", np.nan),
        "validation_accuracy": np.nan,
        "validation_f1": np.nan,
        "validation_roc_auc": np.nan,
        "diagnostic_only": r.get("diagnostic_only", False),
        "note": "Diagnostic-only water transfer; not a primary claim" if r["target"] == "water_saturation" else "Single-well transfer: trained on calibration well only, selected by training-depth-block CV, then tested on external wells",
    })

best_occ = occurrence_result["selected_blind_metrics"].iloc[0].to_dict()
selected_rows.append({
    "target": occurrence_result["target"],
    "task_type": occurrence_result["task_type"],
    "selected_model": occurrence_result["selected_model_name"],
    "feature_set": occurrence_result["feature_set"],
    "validation_well": occurrence_result["validation_well"],
    "validation_wells": occurrence_result["validation_well"],
    "train_wells": "+".join(occurrence_result["train_wells"]),
    "final_train_wells": "+".join(occurrence_result["final_train_wells"]),
    "features_used": ", ".join(occurrence_result["features"]),
    "feature_count": len(occurrence_result["features"]),
    "final_train_rows": occurrence_result["final_train_rows"],
    "validation_rmse": np.nan,
    "validation_mae": np.nan,
    "validation_r2": np.nan,
    "validation_accuracy": best_occ.get("accuracy", np.nan),
    "validation_f1": best_occ.get("f1", np.nan),
    "validation_roc_auc": best_occ.get("roc_auc", np.nan),
    "diagnostic_only": False,
    "note": f"Occurrence label demo: positive Sh >= {OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f}, negative Sh <= {OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f}; trained on calibration well and tested on transfer wells",
})
selected_summary_df = pd.DataFrame(selected_rows)

# Correctness/readiness checks to make the model reviewable before adding stability/producibility.
correctness_checks_df = pd.DataFrame([
    {
        "check": "validation_design",
        "status": f"train {CALIBRATION_TRAIN_WELL} -> test {'+'.join(TRANSFER_TEST_WELLS)}",
        "meaning": "Single-well transfer design: one calibration well trains the model; the other wells are external transfer tests.",
    },
    {
        "check": "training_selection_mode",
        "status": TRAINING_SELECTION_MODE,
        "meaning": "Model/feature-set selection happens inside the training well using depth-block CV, not by choosing the best external transfer result.",
    },
    {
        "check": "normalized_input_mode",
        "status": "active" if NORMALIZED_INPUT_MODE else "inactive",
        "meaning": "All non-depth curves are treated as normalized. Physical equations are proxy/review features unless raw units are supplied.",
    },
    {
        "check": "primary_feature_policy",
        "status": PRIMARY_FEATURE_POLICY,
        "meaning": "Primary model selection is limited to measured/safe-normalized feature sets; proxy equation features are review-only by default.",
    },
    {
        "check": "target_leakage_block",
        "status": "active",
        "meaning": "S_h/S_wr, Archie Sh/Sw, NMR-density Sh, rule screens, labels, and model outputs are blocked from X predictors.",
    },
    {
        "check": "target_bin_weights",
        "status": "active" if APPLY_TARGET_BIN_WEIGHTS else "inactive",
        "meaning": "High-saturation and rare target bins receive weighting so the model is not optimized only for low-saturation rows.",
    },
    {
        "check": "water_model_status",
        "status": WATER_MODEL_MODE,
        "meaning": "Water saturation is not treated as a primary accuracy claim because external water-target coverage is limited.",
    },
])

print("Selected models:")
display(selected_summary_df)
print("Training-well depth-block CV selection metrics:")
display(cv_metrics_df.sort_values(["target", "task_type", "primary_eligible"], ascending=[True, True, False]).head(30))
print("External transfer regression metrics:")
display(metrics_df.sort_values(["target", "selection_stage", "rmse"]).head(30))
print("Per-transfer-well metrics:")
display(transfer_metrics_by_well_df.head(30))
print("Occurrence classifier external transfer metrics:")
display(occurrence_metrics_df.head(20))
print("Correctness checks:")
display(correctness_checks_df)


## 3B. Chong-style ANN replication/extension suite

This cell adds the source-matched ANN layer: six core well-log families, ANN hyperparameter sensitivity, repeated realizations, and an equation-integrated extension.


In [ ]:

# =============================================================================
# V10 Chong et al.-style ANN + geomechanics replication/extension suite
# =============================================================================
# This cell adds a dedicated ANN experiment track instead of relying on sklearn's
# MLPRegressor. It implements the reported ANN structure: ReLU hidden layers,
# linear output, dropout, Adam optimizer, and MSE loss. It also runs repeated
# realizations so we can report R2 distributions, not just a single seed.

CHONG_ANN_RESULTS_XLSX = OUTPUT_DIR / f"chong_ann_results_{RUN_SLUG}.xlsx"
CHONG_ANN_WLC_SUMMARY_CSV = OUTPUT_DIR / f"chong_ann_wlc_summary_{RUN_SLUG}.csv"
CHONG_ANN_REALIZATION_METRICS_CSV = OUTPUT_DIR / f"chong_ann_realization_metrics_{RUN_SLUG}.csv"
CHONG_ANN_SELECTED_PREDICTIONS_CSV = OUTPUT_DIR / f"chong_ann_selected_predictions_{RUN_SLUG}.csv"

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    CHONG_TORCH_AVAILABLE = True
    CHONG_TORCH_IMPORT_ERROR = ""
except Exception as exc:
    CHONG_TORCH_AVAILABLE = False
    CHONG_TORCH_IMPORT_ERROR = str(exc)

CHONG_CORE_FEATURE_MAP = {
    "rho": "rhob_g_cc",
    "phi": "density_porosity_vv",
    "GR": "gr_api",
    "Rt": "rt_ohm_m",
    "Vp": "vp_m_s",
    "Vs": "vs_m_s",
}

# Equation features are the V8/V9 extension retained by V10; target-derived saturation equations
# are still blocked. Archie Sh/Sw and NMR-density Sh stay review/target-side.
CHONG_EQUATION_FEATURE_SETS = {
    "EQ_full_no_target_leakage": [
        "rhob_g_cc", "density_porosity_vv", "gr_api", "rt_ohm_m", "vp_m_s", "vs_m_s",
        "log10_rt", "clean_sand_score", "phi_density_calc", "phi_effective_for_equations",
        "vsh_larionov_tertiary", "vp_vs_ratio", "acoustic_impedance", "shear_impedance",
        "shear_modulus_gpa", "bulk_modulus_gpa", "youngs_modulus_gpa", "youngs_modulus_mpsi",
        "poisson_ratio", "lambda_rho", "mu_rho", "lr_term", "mr_term",
        "brittleness_youngs_pct", "brittleness_poisson_pct", "brittleness_total_pct",
    ],
    "EQ_res_vel_elastic": [
        "phi_effective_for_equations", "log10_rt", "clean_sand_score", "vp_m_s", "vs_m_s",
        "vp_vs_ratio", "acoustic_impedance", "shear_impedance", "mu_rho", "lambda_rho",
        "lr_term", "mr_term", "brittleness_total_pct",
    ],
    "EQ_phi_rt_vp_elastic": [
        "phi_effective_for_equations", "log10_rt", "vp_m_s", "vp_vs_ratio",
        "acoustic_impedance", "bulk_modulus_gpa", "youngs_modulus_gpa", "poisson_ratio",
        "mu_rho", "brittleness_youngs_pct", "brittleness_poisson_pct",
    ],
}

CHONG_BLOCKED_FOR_ANN = set(BLOCKED_FEATURES) | {
    "sh_archie_calc", "sw_archie_calc", "sh_nmr_density_calc",
    "hydrate_occurrence_label", "occurrence_probability_screen", "occurrence_probability_ml",
    "prediction", "prediction_raw", "predicted_label", "residual", "label_residual",
}

CHONG_STYLE_SPLITS = []
if CHONG_RUN_CURRENT_SINGLE_WELL_TRANSFER:
    CHONG_STYLE_SPLITS.append({
        "split_name": "current_single_well_WellC_to_ABD",
        "train_wells": ["WellC"],
        "test_wells": ["WellA", "WellB", "WellD"],
        "split_family": "single_well_transfer_current_project",
        "caveat": "Project stress test: one calibration well trains; all other wells are external transfer wells.",
    })
if CHONG_RUN_BASIN_TRANSFER_ANALOGS:
    CHONG_STYLE_SPLITS.extend([
        {
            "split_name": "partial_ANS_to_Mallik_WellCD_to_AB",
            "train_wells": ["WellC", "WellD"],
            "test_wells": ["WellA", "WellB"],
            "split_family": "chong_style_basin_transfer_partial",
            "caveat": "Chong-style ANS->Mallik analogue using only the active ANS wells in this four-well project runtime.",
        },
        {
            "split_name": "Mallik_to_partial_ANS_WellAB_to_CD",
            "train_wells": ["WellA", "WellB"],
            "test_wells": ["WellC", "WellD"],
            "split_family": "chong_style_basin_transfer",
            "caveat": "Chong-style Mallik->ANS analogue using the two available ANS wells.",
        },
    ])

CHONG_PROTOCOL_ROWS = [
    {"item": "ann_architecture", "implementation": "ReLU hidden layers, linear output layer, dropout after each hidden layer, Adam optimizer, MSE loss."},
    {"item": "reported_fixed_params", "implementation": str(CHONG_FIXED_ANN_PARAMS)},
    {"item": "hyperparameter_sensitivity", "implementation": "Grid over learning rate, hidden layers, nodes/layer, batch size, epochs, dropout."},
    {"item": "realizations", "implementation": f"{CHONG_ANN_REALIZATIONS} repeated ANN random initializations per WLC in {CHONG_RUNTIME_MODE} mode."},
    {"item": "active_project_wells", "implementation": ACTIVE_WELL_SCOPE_NOTE},
    {"item": "header_contracts", "implementation": SOURCE_HEADER_EVIDENCE_NOTE},
    {"item": "core_wlc_features", "implementation": "rho, phi, GR, Rt, Vp, Vs mapped to canonical workbook fields; phi maps to source density_porosity_vv when present."},
    {"item": "chong_phi_policy", "implementation": DENSITY_POROSITY_SOURCE_POLICY},
    {"item": "literature_scope", "implementation": LITERATURE_SCOPE_NOTE},
    {"item": "equation_extension", "implementation": "Equation-derived non-target features are included in separate EQ WLCs; target-derived Sh/Sw equations remain blocked."},
    {"item": "normalized_input_caveat", "implementation": "All non-depth workbook curves are treated as normalized inputs; equation features are relative/proxy ML features unless raw physical units are supplied."},
    {"item": "outlier_caveat", "implementation": "Caliper/QC flags are retained; exact GLOSS implementation is not available, so GLOSS is not claimed as reproduced."},
]
chong_ann_protocol_df = pd.DataFrame(CHONG_PROTOCOL_ROWS)


def chong_available_core_features(df: pd.DataFrame) -> list[tuple[str, str]]:
    out = []
    for label, col in CHONG_CORE_FEATURE_MAP.items():
        if col in df.columns and col not in CHONG_BLOCKED_FOR_ANN and df[col].notna().any():
            out.append((label, col))
    return out


def build_chong_wlc_catalog(df: pd.DataFrame) -> pd.DataFrame:
    core = chong_available_core_features(df)
    rows = []
    reported_wlcs = [
        ("Vp",), ("Rt",), ("Vs",), ("GR",), ("phi",), ("rho",),
        ("phi", "Vp"), ("GR", "Vp"), ("Vp", "Vs"), ("Rt", "Vp"), ("GR", "Vs"),
        ("phi", "Rt", "Vp"), ("GR", "Rt", "Vp"), ("phi", "GR", "Vp"),
        ("phi", "Vp", "Vs"), ("Rt", "Vp", "Vs"), ("GR", "Vp", "Vs"),
    ]
    label_to_col = dict(core)
    if CHONG_WLC_MODE == "all_single_pair_triplet":
        combos = []
        for k in [1, 2, 3]:
            combos.extend(list(combinations([x[0] for x in core], k)))
    else:
        combos = [c for c in reported_wlcs if all(lbl in label_to_col for lbl in c)]
    for combo in combos:
        cols = [label_to_col[lbl] for lbl in combo]
        rows.append({
            "wlc_name": "+".join(combo),
            "wlc_family": "Chong_core_comparable",
            "feature_labels": ", ".join(combo),
            "feature_columns": ", ".join(cols),
            "feature_count": len(cols),
            "is_reported_in_chong_tables": combo in reported_wlcs,
        })
    if CHONG_RUN_EQUATION_INTEGRATED_EXTENSION:
        for name, cols0 in CHONG_EQUATION_FEATURE_SETS.items():
            cols = unique_preserve_order([c for c in cols0 if c in df.columns and c not in CHONG_BLOCKED_FOR_ANN and df[c].notna().any()])
            if len(cols) >= 2:
                rows.append({
                    "wlc_name": name,
                    "wlc_family": "Equation_integrated_extension",
                    "feature_labels": name,
                    "feature_columns": ", ".join(cols),
                    "feature_count": len(cols),
                    "is_reported_in_chong_tables": False,
                })
    return pd.DataFrame(rows).drop_duplicates(subset=["wlc_name", "feature_columns"]).reset_index(drop=True)


def parse_feature_columns(feature_columns: str) -> list[str]:
    return [c.strip() for c in str(feature_columns).split(",") if c.strip()]


def chong_filtered_rows(df: pd.DataFrame, wells: list[str], features: list[str], target_col: str = "hydrate_saturation_reference") -> pd.DataFrame:
    sub = df[df["well_alias"].isin(wells)].copy()
    sub = sub[sub[target_col].notna()].copy()
    features = [f for f in features if f in sub.columns and f not in CHONG_BLOCKED_FOR_ANN]
    if not features:
        return sub.iloc[0:0].copy()
    sub = sub.replace([np.inf, -np.inf], np.nan)
    sub = sub[row_completeness_mask(sub, features)].copy()
    return sub


class ChongANNRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden_layers: int = 2, nodes_per_layer: int = 40, dropout: float = 0.5):
        super().__init__()
        layers = []
        last = input_dim
        for _ in range(int(hidden_layers)):
            layers.append(nn.Linear(last, int(nodes_per_layer)))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(float(dropout)))
            last = int(nodes_per_layer)
        layers.append(nn.Linear(last, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)


def chong_fit_predict_ann(train: pd.DataFrame, valid: pd.DataFrame, features: list[str], params: dict[str, Any], seed: int) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    if not CHONG_TORCH_AVAILABLE:
        raise ImportError(f"PyTorch is required for Chong-style dropout ANN. Import error: {CHONG_TORCH_IMPORT_ERROR}")
    torch.manual_seed(int(seed))
    np.random.seed(int(seed))
    imputer = SimpleImputer(strategy="median")
    scaler = MinMaxScaler()
    Xtr = scaler.fit_transform(imputer.fit_transform(train[features]))
    Xva = scaler.transform(imputer.transform(valid[features]))
    ytr = pd.to_numeric(train["hydrate_saturation_reference"], errors="coerce").astype(float).clip(0, 1).to_numpy()
    yva = pd.to_numeric(valid["hydrate_saturation_reference"], errors="coerce").astype(float).clip(0, 1).to_numpy()
    model = ChongANNRegressor(
        input_dim=Xtr.shape[1],
        hidden_layers=int(params.get("hidden_layers", 2)),
        nodes_per_layer=int(params.get("nodes_per_layer", 40)),
        dropout=float(params.get("dropout", 0.5)),
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=float(params.get("learning_rate", 0.001)))
    loss_fn = nn.MSELoss()
    batch_size = int(params.get("batch_size", 100))
    epochs = int(params.get("epochs", 500))
    ds = TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.float32))
    loader = DataLoader(ds, batch_size=max(1, min(batch_size, len(ds))), shuffle=True)
    model.train()
    final_loss = np.nan
    for _epoch in range(epochs):
        losses = []
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu().item()))
        final_loss = float(np.mean(losses)) if losses else np.nan
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(Xva, dtype=torch.float32)).detach().cpu().numpy()
    pred = np.clip(pred, 0.0, 1.0)
    fitted = {"imputer": imputer, "scaler": scaler, "model": model, "params": dict(params), "seed": seed, "train_loss": final_loss}
    return yva, pred, fitted


def chong_metrics(y_true: np.ndarray, pred: np.ndarray) -> dict[str, float]:
    return {
        "n": int(len(y_true)),
        "rmse": float(math.sqrt(mean_squared_error(y_true, pred))) if len(y_true) else np.nan,
        "mae": float(mean_absolute_error(y_true, pred)) if len(y_true) else np.nan,
        "r2": float(r2_score(y_true, pred)) if len(y_true) > 1 else np.nan,
        "bias": float(np.mean(pred - y_true)) if len(y_true) else np.nan,
        "prediction_min": float(np.min(pred)) if len(pred) else np.nan,
        "prediction_max": float(np.max(pred)) if len(pred) else np.nan,
    }


def hparam_grid_records() -> list[dict[str, Any]]:
    rows = []
    for lr, layers, nodes, batch, epochs, dropout in product(CHONG_GRID_LEARNING_RATES, CHONG_GRID_HIDDEN_LAYERS, CHONG_GRID_NODES_PER_LAYER, CHONG_GRID_BATCH_SIZES, CHONG_GRID_EPOCHS, CHONG_GRID_DROPOUT):
        rows.append({"learning_rate": lr, "hidden_layers": layers, "nodes_per_layer": nodes, "batch_size": batch, "epochs": epochs, "dropout": dropout})
    if CHONG_GRID_SAMPLE_LIMIT is not None:
        rows = rows[:int(CHONG_GRID_SAMPLE_LIMIT)]
    return rows


def run_chong_hparam_sensitivity(df: pd.DataFrame, split: dict[str, Any], tuning_features: list[str]) -> pd.DataFrame:
    rows = []
    train_all = chong_filtered_rows(df, split["train_wells"], tuning_features)
    if len(train_all) < 30 or len(tuning_features) < 1:
        return pd.DataFrame([{"split_name": split["split_name"], "status": "skipped_not_enough_training_rows_or_features", "train_rows": len(train_all), "feature_count": len(tuning_features)}])
    splitter = ShuffleSplit(n_splits=CHONG_TUNING_SPLITS, test_size=0.20, random_state=RANDOM_SEED)
    grid = hparam_grid_records()
    t0 = time.time()
    for combo_i, params in enumerate(grid, start=1):
        fold_metrics = []
        for fold_i, (tr_idx, va_idx) in enumerate(splitter.split(train_all), start=1):
            tr = train_all.iloc[tr_idx].copy()
            va = train_all.iloc[va_idx].copy()
            try:
                y, p, fitted = chong_fit_predict_ann(tr, va, tuning_features, params, seed=RANDOM_SEED + 1000 * combo_i + fold_i)
                m = chong_metrics(y, p)
                m.update({"fold": fold_i, "train_loss": fitted.get("train_loss", np.nan), "status": "ok"})
            except Exception as exc:
                m = {"fold": fold_i, "rmse": np.nan, "mae": np.nan, "r2": np.nan, "status": "failed", "note": str(exc)}
            fold_metrics.append(m)
        fdf = pd.DataFrame(fold_metrics)
        row = dict(params)
        row.update({
            "split_name": split["split_name"], "split_family": split.get("split_family", ""),
            "feature_set": "all_six_core_logs_for_hparam_tuning", "feature_columns": ", ".join(tuning_features),
            "train_rows": len(train_all), "cv_mode": "Chong_style_random_80_20_ShuffleSplit",
            "cv_folds": int(fdf["r2"].notna().sum()) if "r2" in fdf else 0,
            "cv_r2_mean": float(fdf["r2"].mean()) if "r2" in fdf else np.nan,
            "cv_r2_std": float(fdf["r2"].std()) if "r2" in fdf else np.nan,
            "cv_rmse_mean": float(fdf["rmse"].mean()) if "rmse" in fdf else np.nan,
            "cv_mae_mean": float(fdf["mae"].mean()) if "mae" in fdf else np.nan,
            "status": "ok" if fdf.get("r2", pd.Series(dtype=float)).notna().any() else "failed",
            "elapsed_seconds_so_far": round(time.time() - t0, 2),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def pearson_realization_summary(pred_matrix: np.ndarray) -> dict[str, float]:
    if pred_matrix.ndim != 2 or pred_matrix.shape[0] < 2:
        return {"pearson_pair_mean": np.nan, "pearson_pair_median": np.nan, "pearson_pair_std": np.nan, "pearson_pair_max": np.nan}
    corr = np.corrcoef(pred_matrix)
    tri = corr[np.triu_indices_from(corr, k=1)]
    tri = tri[np.isfinite(tri)]
    if len(tri) == 0:
        return {"pearson_pair_mean": np.nan, "pearson_pair_median": np.nan, "pearson_pair_std": np.nan, "pearson_pair_max": np.nan}
    return {"pearson_pair_mean": float(np.mean(tri)), "pearson_pair_median": float(np.median(tri)), "pearson_pair_std": float(np.std(tri)), "pearson_pair_max": float(np.max(tri))}


def run_chong_repeated_wlc_suite(df: pd.DataFrame, split: dict[str, Any], catalog: pd.DataFrame, fixed_params: dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    metric_rows, summary_rows, pearson_rows, selected_pred_tables = [], [], [], []
    if not CHONG_TORCH_AVAILABLE:
        msg = f"PyTorch unavailable: {CHONG_TORCH_IMPORT_ERROR}"
        return (pd.DataFrame([{"split_name": split["split_name"], "status": "failed", "note": msg}]), pd.DataFrame([{"split_name": split["split_name"], "status": "failed", "note": msg}]), pd.DataFrame([{"split_name": split["split_name"], "status": "failed", "note": msg}]), pd.DataFrame())
    for _, row in catalog.iterrows():
        wlc_name = row["wlc_name"]
        features = parse_feature_columns(row["feature_columns"])
        train = chong_filtered_rows(df, split["train_wells"], features)
        valid = chong_filtered_rows(df, split["test_wells"], features)
        if len(train) < 20 or len(valid) < 5 or len(features) < 1:
            summary_rows.append({"split_name": split["split_name"], "wlc_name": wlc_name, "wlc_family": row["wlc_family"], "status": "skipped_not_enough_rows_or_features", "train_rows": len(train), "test_rows": len(valid), "feature_count": len(features)})
            continue
        pred_matrix = []
        valid_meta = valid[[c for c in ["well_alias", "well_name", "site", "depth_m", "depth_ft", "hydrate_saturation_reference", "qc_status"] if c in valid.columns]].copy()
        for realization in range(1, CHONG_ANN_REALIZATIONS + 1):
            seed = RANDOM_SEED + realization * 37 + abs(hash((split["split_name"], wlc_name))) % 10000
            try:
                y, pred, fitted = chong_fit_predict_ann(train, valid, features, fixed_params, seed=seed)
                m = chong_metrics(y, pred)
                m.update({
                    "split_name": split["split_name"], "split_family": split.get("split_family", ""), "train_wells": wells_label(split["train_wells"]), "test_wells": wells_label(split["test_wells"]),
                    "wlc_name": wlc_name, "wlc_family": row["wlc_family"], "feature_labels": row["feature_labels"], "feature_columns": row["feature_columns"], "feature_count": row["feature_count"],
                    "realization": realization, "seed": seed, "ann_hidden_layers": fixed_params["hidden_layers"], "ann_nodes_per_layer": fixed_params["nodes_per_layer"], "ann_learning_rate": fixed_params["learning_rate"],
                    "ann_batch_size": fixed_params["batch_size"], "ann_epochs": fixed_params["epochs"], "ann_dropout": fixed_params["dropout"], "train_rows": len(train), "test_rows": len(valid),
                    "train_loss": fitted.get("train_loss", np.nan), "status": "ok",
                })
                metric_rows.append(m)
                pred_matrix.append(pred)
            except Exception as exc:
                metric_rows.append({"split_name": split["split_name"], "wlc_name": wlc_name, "wlc_family": row["wlc_family"], "realization": realization, "status": "failed", "note": str(exc)})
        mdf = pd.DataFrame([r for r in metric_rows if r.get("split_name") == split["split_name"] and r.get("wlc_name") == wlc_name and r.get("status") == "ok"])
        if len(mdf):
            pmat = np.vstack(pred_matrix) if pred_matrix else np.empty((0, len(valid)))
            pr = pearson_realization_summary(pmat)
            best_idx = int(mdf["r2"].idxmax()) if mdf["r2"].notna().any() else mdf.index[0]
            best_realization = int(mdf.loc[best_idx, "realization"])
            summary = {
                "split_name": split["split_name"], "split_family": split.get("split_family", ""), "train_wells": wells_label(split["train_wells"]), "test_wells": wells_label(split["test_wells"]),
                "wlc_name": wlc_name, "wlc_family": row["wlc_family"], "feature_labels": row["feature_labels"], "feature_columns": row["feature_columns"], "feature_count": row["feature_count"],
                "is_reported_in_chong_tables": row.get("is_reported_in_chong_tables", False), "realization_count": int(len(mdf)), "r2_mean": float(mdf["r2"].mean()), "r2_std": float(mdf["r2"].std()),
                "r2_min": float(mdf["r2"].min()), "r2_max": float(mdf["r2"].max()), "rmse_mean": float(mdf["rmse"].mean()), "mae_mean": float(mdf["mae"].mean()), "bias_mean": float(mdf["bias"].mean()),
                "best_realization": best_realization, "status": "ok", "caveat": split.get("caveat", ""),
            }
            summary.update(pr)
            summary_rows.append(summary)
            pearson_rows.append(dict(summary))
            if len(pred_matrix):
                best_pos = best_realization - 1
                if 0 <= best_pos < len(pred_matrix):
                    tab = valid_meta.copy()
                    tab["split_name"] = split["split_name"]
                    tab["wlc_name"] = wlc_name
                    tab["wlc_family"] = row["wlc_family"]
                    tab["selected_realization"] = best_realization
                    tab["reference"] = pd.to_numeric(valid["hydrate_saturation_reference"], errors="coerce").astype(float).clip(0, 1).to_numpy()
                    tab["prediction"] = pred_matrix[best_pos]
                    tab["residual"] = tab["prediction"] - tab["reference"]
                    selected_pred_tables.append(tab)
    metrics_df_local = pd.DataFrame(metric_rows)
    summary_df_local = pd.DataFrame(summary_rows)
    pearson_df_local = pd.DataFrame(pearson_rows)
    preds_df_local = pd.concat(selected_pred_tables, ignore_index=True) if selected_pred_tables else pd.DataFrame()
    if len(summary_df_local) and len(preds_df_local):
        top_keys = summary_df_local[summary_df_local["status"].eq("ok")].sort_values("r2_mean", ascending=False).head(CHONG_SAVE_TOP_PREDICTION_TABLES)[["split_name", "wlc_name"]]
        keep = set(map(tuple, top_keys.to_numpy()))
        preds_df_local = preds_df_local[preds_df_local.apply(lambda r: (r["split_name"], r["wlc_name"]) in keep, axis=1)].copy()
    return metrics_df_local, summary_df_local, pearson_df_local, preds_df_local


chong_ann_wlc_catalog_df = build_chong_wlc_catalog(features_df)
core_features_available = [c for _, c in chong_available_core_features(features_df)]
all_hparam_tables, all_metric_tables, all_summary_tables, all_pearson_tables, all_prediction_tables = [], [], [], [], []

if not CHONG_TORCH_AVAILABLE:
    chong_ann_hparam_sensitivity_df = pd.DataFrame([{"status": "failed", "note": f"PyTorch unavailable: {CHONG_TORCH_IMPORT_ERROR}"}])
    chong_ann_realization_metrics_df = pd.DataFrame([{"status": "failed", "note": f"PyTorch unavailable: {CHONG_TORCH_IMPORT_ERROR}"}])
    chong_ann_wlc_summary_df = pd.DataFrame([{"status": "failed", "note": f"PyTorch unavailable: {CHONG_TORCH_IMPORT_ERROR}"}])
    chong_ann_pearson_df = pd.DataFrame([{"status": "failed", "note": f"PyTorch unavailable: {CHONG_TORCH_IMPORT_ERROR}"}])
    chong_ann_selected_predictions_df = pd.DataFrame()
else:
    for split in CHONG_STYLE_SPLITS:
        print(f"Running Chong-style ANN sensitivity for {split['split_name']} with {CHONG_ANN_REALIZATIONS} realization(s) per WLC...")
        if len(core_features_available) >= 2:
            all_hparam_tables.append(run_chong_hparam_sensitivity(features_df, split, core_features_available))
        m, s, p, predtab = run_chong_repeated_wlc_suite(features_df, split, chong_ann_wlc_catalog_df, CHONG_FIXED_ANN_PARAMS)
        all_metric_tables.append(m); all_summary_tables.append(s); all_pearson_tables.append(p)
        if len(predtab): all_prediction_tables.append(predtab)
    chong_ann_hparam_sensitivity_df = pd.concat(all_hparam_tables, ignore_index=True) if all_hparam_tables else pd.DataFrame()
    chong_ann_realization_metrics_df = pd.concat(all_metric_tables, ignore_index=True) if all_metric_tables else pd.DataFrame()
    chong_ann_wlc_summary_df = pd.concat(all_summary_tables, ignore_index=True) if all_summary_tables else pd.DataFrame()
    chong_ann_pearson_df = pd.concat(all_pearson_tables, ignore_index=True) if all_pearson_tables else pd.DataFrame()
    chong_ann_selected_predictions_df = pd.concat(all_prediction_tables, ignore_index=True) if all_prediction_tables else pd.DataFrame()

try:
    chong_ann_wlc_summary_df.to_csv(CHONG_ANN_WLC_SUMMARY_CSV, index=False)
    chong_ann_realization_metrics_df.to_csv(CHONG_ANN_REALIZATION_METRICS_CSV, index=False)
    if len(chong_ann_selected_predictions_df): chong_ann_selected_predictions_df.to_csv(CHONG_ANN_SELECTED_PREDICTIONS_CSV, index=False)
    with pd.ExcelWriter(CHONG_ANN_RESULTS_XLSX, engine="openpyxl") as writer:
        chong_ann_protocol_df.to_excel(writer, sheet_name="ann_protocol", index=False)
        chong_ann_wlc_catalog_df.to_excel(writer, sheet_name="wlc_catalog", index=False)
        chong_ann_hparam_sensitivity_df.to_excel(writer, sheet_name="hparam_sensitivity", index=False)
        chong_ann_wlc_summary_df.to_excel(writer, sheet_name="wlc_summary", index=False)
        chong_ann_pearson_df.to_excel(writer, sheet_name="pearson_precision", index=False)
        chong_ann_realization_metrics_df.to_excel(writer, sheet_name="realization_metrics", index=False)
        if len(chong_ann_selected_predictions_df): chong_ann_selected_predictions_df.to_excel(writer, sheet_name="selected_predictions", index=False)
except PermissionError:
    fallback = CHONG_ANN_RESULTS_XLSX.with_name(f"{CHONG_ANN_RESULTS_XLSX.stem}_{RUN_ID}{CHONG_ANN_RESULTS_XLSX.suffix}")
    with pd.ExcelWriter(fallback, engine="openpyxl") as writer:
        chong_ann_protocol_df.to_excel(writer, sheet_name="ann_protocol", index=False)
        chong_ann_wlc_catalog_df.to_excel(writer, sheet_name="wlc_catalog", index=False)
        chong_ann_hparam_sensitivity_df.to_excel(writer, sheet_name="hparam_sensitivity", index=False)
        chong_ann_wlc_summary_df.to_excel(writer, sheet_name="wlc_summary", index=False)
        chong_ann_pearson_df.to_excel(writer, sheet_name="pearson_precision", index=False)
        chong_ann_realization_metrics_df.to_excel(writer, sheet_name="realization_metrics", index=False)
        if len(chong_ann_selected_predictions_df): chong_ann_selected_predictions_df.to_excel(writer, sheet_name="selected_predictions", index=False)
    CHONG_ANN_RESULTS_XLSX = fallback

print("Chong ANN protocol:")
display(chong_ann_protocol_df)
print("Chong WLC catalog:")
display(chong_ann_wlc_catalog_df.head(30))
print("Chong ANN WLC summary:")
display(chong_ann_wlc_summary_df.sort_values(["split_name", "r2_mean"], ascending=[True, False]).head(30) if len(chong_ann_wlc_summary_df) and "r2_mean" in chong_ann_wlc_summary_df.columns else chong_ann_wlc_summary_df.head(30))
print("Dedicated Chong ANN workbook:", CHONG_ANN_RESULTS_XLSX)


## 4. Export consolidated V10 outputs

In [ ]:
# =============================================================================
# Consolidated outputs — V9 single-well transfer workbook
# =============================================================================

OUTPUT_XLSX = OUTPUT_DIR / f"model_results_{RUN_SLUG}.xlsx"
PREDICTIONS_CSV = OUTPUT_DIR / f"predictions_{RUN_SLUG}.csv"
SELECTED_PREDICTIONS_CSV = OUTPUT_DIR / f"selected_predictions_{RUN_SLUG}.csv"
OCCURRENCE_PREDICTIONS_CSV = OUTPUT_DIR / f"occurrence_predictions_{RUN_SLUG}.csv"
FIGURES_PDF = OUTPUT_DIR / f"paper_figures_{RUN_SLUG}.pdf"
MANIFEST_JSON = OUTPUT_DIR / f"run_manifest_{RUN_SLUG}.json"
MODEL_JOBLIB = MODEL_DIR / f"selected_models_{RUN_SLUG}.joblib"
ACTIVE_WELL_POLICY_CSV = OUTPUT_DIR / f"active_well_policy_{RUN_SLUG}.csv"
HEADER_CONTRACT_SUMMARY_CSV = OUTPUT_DIR / f"header_contract_summary_{RUN_SLUG}.csv"
DENSITY_POROSITY_POLICY_CSV = OUTPUT_DIR / f"density_porosity_policy_summary_{RUN_SLUG}.csv"
CHONG_CORE_FEATURE_PRESENCE_CSV = OUTPUT_DIR / f"chong_core_feature_presence_by_well_{RUN_SLUG}.csv"
FEATURE_POLICY_SUMMARY_CSV = OUTPUT_DIR / f"feature_policy_summary_{RUN_SLUG}.csv"
# V10 Chong ANN outputs are created in the ANN suite cell and added to the main workbook when present.

hydrate_validation_label = "+".join(HYDRATE_VALIDATION_WELLS)
occurrence_validation_label = "+".join(OCCURRENCE_VALIDATION_WELLS)
water_validation_label = "+".join(WATER_VALIDATION_WELLS)

# Compact well summary.
well_summary = features_df.groupby(["well_alias", "well_name", "site"], dropna=False).agg(
    rows=("well_alias", "size"),
    hydrate_target_rows=("hydrate_saturation_reference", lambda x: int(x.notna().sum())),
    occurrence_labeled_rows=("hydrate_occurrence_label", lambda x: int(x.notna().sum())),
    occurrence_positive_rows=("hydrate_occurrence_label", lambda x: int((x == 1).sum())),
    occurrence_negative_rows=("hydrate_occurrence_label", lambda x: int((x == 0).sum())),
    water_target_rows=("water_saturation_reference", lambda x: int(x.notna().sum())),
    rt_coverage=("rt_ohm_m", lambda x: float(x.notna().mean())),
    vp_coverage=("vp_m_s", lambda x: float(x.notna().mean())),
    vs_coverage=("vs_m_s", lambda x: float(x.notna().mean())),
    qc_review_rows=("qc_status", lambda x: int((x == "review").sum())),
    rw_est_ohm_m=("rw_est_ohm_m", lambda x: float(x.dropna().median()) if x.notna().any() else np.nan),
 ).reset_index()

loaded_well_aliases = sorted(features_df["well_alias"].dropna().unique().tolist())
active_well_policy_df = pd.DataFrame([
    {
        "well_alias": alias,
        "well_name": meta["well_name"],
        "site": meta["site"],
        "source_layout": meta["layout"],
        "source_header_contract": meta.get("source_header_contract", ""),
        "loaded_in_runtime": alias in loaded_well_aliases,
        "active_scope_note": ACTIVE_WELL_SCOPE_NOTE,
    }
    for alias, meta in WELL_METADATA.items()
])

header_contract_summary_df = pd.DataFrame([
    {
        "source_header_contract": key,
        "layout": value.get("layout", ""),
        "scope": value.get("scope", ""),
        "screenshots": "; ".join(value.get("screenshots", [])),
        "visible_headers": "; ".join(value.get("visible_headers", [])),
        "active_wells": "+".join([alias for alias, meta in WELL_METADATA.items() if meta.get("source_header_contract") == key]),
    }
    for key, value in HEADER_CONTRACTS.items()
])


def mapping_value(alias: str, canonical_field: str, column: str, default: Any = "") -> Any:
    sub = field_mapping[(field_mapping["well_alias"] == alias) & (field_mapping["canonical_field"] == canonical_field)]
    if sub.empty or column not in sub.columns:
        return default
    value = sub.iloc[0][column]
    if pd.isna(value):
        return default
    return value


def coverage_for(group: pd.DataFrame, column: str) -> float:
    if column not in group.columns or len(group) == 0:
        return float("nan")
    return float(group[column].notna().mean())


def source_count_json(series: pd.Series) -> str:
    counts = {str(key): int(value) for key, value in series.value_counts(dropna=False).items()}
    return json.dumps(counts, sort_keys=True)


density_porosity_policy_summary = pd.DataFrame([
    {
        "well_alias": alias,
        "well_name": WELL_METADATA[alias]["well_name"],
        "source_layout": WELL_METADATA[alias]["layout"],
        "source_header_contract": WELL_METADATA[alias].get("source_header_contract", ""),
        "source_density_porosity_header": mapping_value(alias, "density_porosity_vv", "source_header"),
        "source_density_porosity_coverage": coverage_for(group, "density_porosity_vv"),
        "rhob_header": mapping_value(alias, "rhob_g_cc", "source_header"),
        "rhob_coverage": coverage_for(group, "rhob_g_cc"),
        "phi_density_calc_coverage": coverage_for(group, "phi_density_calc"),
        "phi_effective_coverage": coverage_for(group, "phi_effective_for_equations"),
        "phi_effective_source_counts": source_count_json(group["phi_effective_source"]),
        "chong_phi_column": CHONG_CORE_FEATURE_MAP.get("phi", "density_porosity_vv") if "CHONG_CORE_FEATURE_MAP" in globals() else "density_porosity_vv",
        "policy": DENSITY_POROSITY_SOURCE_POLICY,
    }
    for alias, group in features_df.groupby("well_alias", dropna=False)
])

chong_core_feature_presence_by_well = pd.DataFrame([
    {
        "well_alias": alias,
        "well_name": WELL_METADATA[alias]["well_name"],
        "chong_label": label,
        "canonical_column": column,
        "source_header": mapping_value(alias, column, "source_header"),
        "coverage": coverage_for(group, column),
        "source_header_contract": WELL_METADATA[alias].get("source_header_contract", ""),
    }
    for alias, group in features_df.groupby("well_alias", dropna=False)
    for label, column in (CHONG_CORE_FEATURE_MAP.items() if "CHONG_CORE_FEATURE_MAP" in globals() else [])
])

feature_policy_summary_df = pd.DataFrame([
    *[{"policy_group": "measured_features", "feature": f, "x_policy": "allowed_if_coverage_passes", "notes": "Raw/supplied log feature after canonical mapping."} for f in MEASURED_FEATURES],
    *[{"policy_group": "safe_transform_features", "feature": f, "x_policy": "allowed_if_coverage_passes", "notes": "Transform of measured logs only."} for f in SAFE_TRANSFORM_FEATURES],
    *[{"policy_group": "proxy_equation_features", "feature": f, "x_policy": "allowed_in_equation_extension", "notes": "Physics-derived feature; target-derived saturation equations excluded."} for f in PROXY_EQUATION_FEATURES],
    *[{"policy_group": "context_features", "feature": f, "x_policy": "review_context", "notes": "Context feature controlled by ALLOW_PRESSURE_CONTEXT_IN_PRIMARY_MODEL."} for f in CONTEXT_FEATURES],
    *[{"policy_group": "blocked_features", "feature": f, "x_policy": "blocked_from_x", "notes": "Target, review output, identifier, source metadata, or leakage risk."} for f in sorted(BLOCKED_FEATURES)],
])

occ_counts = features_df.groupby(["well_alias", "hydrate_occurrence_screen"], dropna=False).size().reset_index(name="rows")
occ_pivot = occ_counts.pivot(index="well_alias", columns="hydrate_occurrence_screen", values="rows").fillna(0).reset_index()

occ_label_counts = features_df.groupby(["well_alias", "occurrence_label_status", "hydrate_occurrence_label"], dropna=False).size().reset_index(name="rows")
occurrence_label_rule_df = pd.DataFrame([
    {"item": "positive_label", "rule": f"hydrate_occurrence_label = 1 where hydrate_saturation_reference >= {OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f}", "purpose": "Target-side occurrence label for classifier"},
    {"item": "negative_label", "rule": f"hydrate_occurrence_label = 0 where hydrate_saturation_reference <= {OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f}", "purpose": "Clear non-hydrate / water-like target interval"},
    {"item": "gray_zone", "rule": f"{OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f} < hydrate_saturation_reference < {OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f} omitted from occurrence training", "purpose": "Avoid teaching classifier ambiguous transition labels"},
    {"item": "leakage_guardrail", "rule": "hydrate_occurrence_label and hydrate_saturation_reference are blocked from X_allowed", "purpose": "Classifier learns from logs/safe features only"},
    {"item": "validation_choice", "rule": f"train {'+'.join(OCCURRENCE_TRAIN_WELLS)} -> validate {occurrence_validation_label}", "purpose": "Single-well transfer demonstration"},
])

qc_summary = features_df.groupby(["well_alias", "qc_status", "qc_caliper_status"], dropna=False).agg(
    rows=("well_alias", "size"),
    low_required_log_coverage_rows=("qc_low_required_log_coverage_flag", lambda x: int(x.fillna(False).sum())),
    missing_rt_rows=("qc_missing_rt_flag", lambda x: int(x.fillna(False).sum())),
    missing_velocity_rows=("qc_missing_velocity_flag", lambda x: int(x.fillna(False).sum())),
    missing_porosity_rows=("qc_missing_porosity_flag", lambda x: int(x.fillna(False).sum())),
    bad_caliper_rows=("qc_bad_caliper_flag", lambda x: int(x.fillna(False).sum())),
    elastic_invalid_rows=("qc_elastic_invalid_flag", lambda x: int(x.fillna(False).sum())),
).reset_index()

run_summary_rows = [
    {"section": "run", "name": "run_id", "value": RUN_ID},
    {"section": "run", "name": "code_version", "value": CODE_VERSION},
    {"section": "run", "name": "active_project_wells", "value": "+".join(ACTIVE_PROJECT_WELLS)},
    {"section": "run", "name": "input_dir", "value": str(INPUT_DIR)},
    {"section": "run", "name": "output_dir", "value": str(OUTPUT_DIR)},
    {"section": "run", "name": "model_dir", "value": str(MODEL_DIR)},
    {"section": "policy", "name": "normalized_input_mode", "value": str(NORMALIZED_INPUT_MODE)},
    {"section": "policy", "name": "primary_feature_policy", "value": PRIMARY_FEATURE_POLICY},
    {"section": "policy", "name": "density_porosity_source_policy", "value": DENSITY_POROSITY_SOURCE_POLICY},
    {"section": "policy", "name": "source_header_evidence", "value": SOURCE_HEADER_EVIDENCE_NOTE},
    {"section": "policy", "name": "training_selection_mode", "value": TRAINING_SELECTION_MODE},
    {"section": "policy", "name": "calibration_train_well", "value": CALIBRATION_TRAIN_WELL},
    {"section": "policy", "name": "transfer_test_wells", "value": "+".join(TRANSFER_TEST_WELLS)},
    {"section": "policy", "name": "allow_proxy_equation_features_in_primary_model", "value": str(ALLOW_PROXY_EQUATION_FEATURES_IN_PRIMARY_MODEL)},
    {"section": "policy", "name": "allow_pressure_context_in_primary_model", "value": str(ALLOW_PRESSURE_CONTEXT_IN_PRIMARY_MODEL)},
    {"section": "policy", "name": "deep_resistivity_alias_confirmed", "value": str(DEEP_RESISTIVITY_ALIAS_CONFIRMED)},
    {"section": "policy", "name": "hydrate_split", "value": f"{'+'.join(HYDRATE_TRAIN_WELLS)} -> {hydrate_validation_label}"},
    {"section": "policy", "name": "occurrence_split", "value": f"{'+'.join(OCCURRENCE_TRAIN_WELLS)} -> {occurrence_validation_label}"},
    {"section": "policy", "name": "water_split", "value": f"{'+'.join(WATER_TRAIN_WELLS)} -> {water_validation_label}"},
    {"section": "policy", "name": "water_model_mode", "value": WATER_MODEL_MODE},
    {"section": "policy", "name": "target_bin_weights", "value": str(APPLY_TARGET_BIN_WEIGHTS)},
    {"section": "policy", "name": "occurrence_label_rule", "value": f"1 if Sh >= {OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f}; 0 if Sh <= {OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f}; gray zone omitted"},
    {"section": "policy", "name": "rw_policy", "value": "estimate Rw from S_wr/Swr where available; proxy only in normalized-input mode" if ESTIMATE_RW_FROM_WATER_TARGET else "no Rw estimation"},
    {"section": "policy", "name": "nmr_porosity_feature", "value": str(INCLUDE_NMR_POROSITY_AS_FEATURE)},
]
summary_df = pd.concat([
    pd.DataFrame(run_summary_rows),
    selected_summary_df.assign(section="selected_model").rename(columns={"target": "name", "selected_model": "value"})[["section", "name", "value", "task_type", "feature_set", "validation_well", "train_wells", "validation_rmse", "validation_mae", "validation_r2", "validation_accuracy", "validation_f1", "validation_roc_auc", "diagnostic_only", "note"]],
], ignore_index=True, sort=False)

# Add equation constants and Archie metadata.
archie_meta_df = pd.DataFrame([
    {
        "well_alias": alias,
        "well_name": meta["well_name"],
        "site": meta["site"],
        "lat": meta.get("lat", np.nan),
        "lon": meta.get("lon", np.nan),
        "archie_a": meta.get("archie_a", np.nan),
        "archie_m": meta.get("archie_m", np.nan),
        "archie_n": meta.get("archie_n", np.nan),
        "archie_note": meta.get("archie_note", ""),
        "rw_est_ohm_m": features_df.loc[features_df["well_alias"] == alias, "rw_est_ohm_m"].dropna().median() if features_df.loc[features_df["well_alias"] == alias, "rw_est_ohm_m"].notna().any() else np.nan,
        "hydrate_target_header": meta.get("hydrate_target_header"),
        "water_target_header": meta.get("water_target_header"),
    }
    for alias, meta in WELL_METADATA.items()
])
equations_compact = pd.concat([
    equations_used,
    pd.DataFrame([
        {"field": "density_constants", "equation": "rho_matrix=2.65 g/cc; rho_fluid=1.02 g/cc", "role": "assumption", "predictor_allowed": "n/a", "notes": "Used for density-porosity calculation; proxy caveat applies in normalized-input mode."},
        {"field": "single_well_transfer", "equation": f"hydrate: {'+'.join(HYDRATE_TRAIN_WELLS)} -> {hydrate_validation_label}; occurrence: {'+'.join(OCCURRENCE_TRAIN_WELLS)} -> {occurrence_validation_label}; water: {'+'.join(WATER_TRAIN_WELLS)} -> {water_validation_label}", "role": "validation design", "predictor_allowed": "n/a", "notes": "Train on one calibration well and test on external transfer wells."},
        {"field": "training_depth_block_cv_selection", "equation": "Depth-block CV inside the calibration training well selects model/feature set before external transfer evaluation", "role": "model correctness", "predictor_allowed": "n/a", "notes": "Prevents choosing the best-looking model directly from the external transfer wells."},
        {"field": "primary_feature_policy", "equation": PRIMARY_FEATURE_POLICY, "role": "model correctness", "predictor_allowed": "n/a", "notes": "Measured and safe-normalized features are primary; physical proxy equations are review-only by default."},
        {"field": "deep_resistivity_alias", "equation": ", ".join(DEEP_RESISTIVITY_ALIASES_CONFIRMED), "role": "data dictionary", "predictor_allowed": "n/a", "notes": "Confirmed by project review as deep formation resistivity family."},
        {"field": "density_porosity_source_policy", "equation": DENSITY_POROSITY_SOURCE_POLICY, "role": "data dictionary", "predictor_allowed": "n/a", "notes": "Source density porosity is Chong phi where present; RHOB phi_D is fallback/proxy."},
        {"field": "active_project_wells", "equation": "+".join(ACTIVE_PROJECT_WELLS), "role": "runtime scope", "predictor_allowed": "n/a", "notes": ACTIVE_WELL_SCOPE_NOTE},
    ])
], ignore_index=True)

selected_predictions_df = pd.concat(
    [*[r["selected_predictions"] for r in results], occurrence_result["selected_predictions"]],
    ignore_index=True,
    sort=False,
)

# Write CSVs.
predictions_df.to_csv(PREDICTIONS_CSV, index=False)
selected_predictions_df.to_csv(SELECTED_PREDICTIONS_CSV, index=False)
occurrence_predictions_df.to_csv(OCCURRENCE_PREDICTIONS_CSV, index=False)
active_well_policy_df.to_csv(ACTIVE_WELL_POLICY_CSV, index=False)
header_contract_summary_df.to_csv(HEADER_CONTRACT_SUMMARY_CSV, index=False)
density_porosity_policy_summary.to_csv(DENSITY_POROSITY_POLICY_CSV, index=False)
chong_core_feature_presence_by_well.to_csv(CHONG_CORE_FEATURE_PRESENCE_CSV, index=False)
feature_policy_summary_df.to_csv(FEATURE_POLICY_SUMMARY_CSV, index=False)

# Write figures.
with PdfPages(FIGURES_PDF) as pdf:
    # 1. Selected saturation prediction vs reference and depth profile.
    for r in results:
        target = r["target"]
        selected = r["selected_model_name"]
        feature_set = r["feature_set"]
        plot_df = r["selected_predictions"].copy()
        if len(plot_df):
            fig, ax = plt.subplots(figsize=(6.5, 5.5))
            ax.scatter(plot_df["reference"], plot_df["prediction"], s=12, alpha=0.7)
            ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.set_xlabel("Reference saturation")
            ax.set_ylabel("Predicted saturation")
            m = metric_dict(plot_df["reference"], plot_df["prediction"])
            title_note = "diagnostic only" if r.get("diagnostic_only", False) else "single-well transfer"
            ax.set_title(f"{target}: {selected} / {feature_set}\n{'+'.join(r['train_wells'])} → {r['validation_well']} | RMSE={m['rmse']:.3f}, R²={m['r2']:.3f}, n={m['n']} | {title_note}")
            ax.grid(True, alpha=0.25)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

            for well_alias, well_df in plot_df.groupby("well_alias"):
                fig, ax = plt.subplots(figsize=(7, 7))
                depth = well_df["depth_m"]
                ax.plot(well_df["reference"], depth, label="reference", linewidth=1.8)
                ax.plot(well_df["prediction"], depth, label="prediction", linewidth=1.4)
                ax.set_xlim(0, 1)
                ax.invert_yaxis()
                ax.set_xlabel("Saturation fraction")
                ax.set_ylabel("Depth (m)")
                ax.set_title(f"Depth profile — {target}, transfer {well_alias} ({selected})")
                ax.legend()
                ax.grid(True, alpha=0.25)
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)

    # 2. Training-well CV model-selection comparison for hydrate saturation.
    hydrate_cv = cv_metrics_df[(cv_metrics_df["target"] == "hydrate_saturation") & (cv_metrics_df["cv_rmse_mean"].notna())].copy()
    if len(hydrate_cv):
        hydrate_cv = hydrate_cv.sort_values("cv_rmse_mean").head(14)
        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        labels = hydrate_cv["feature_set"] + "\n" + hydrate_cv["model"]
        ax.bar(np.arange(len(hydrate_cv)), hydrate_cv["cv_rmse_mean"])
        ax.set_xticks(np.arange(len(hydrate_cv)))
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Training-well depth-block CV RMSE")
        ax.set_title("Hydrate model selection happens inside the calibration training well")
        ax.grid(True, axis="y", alpha=0.25)
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

    # 3. External transfer model comparison, transparent candidates.
    comp = metrics_df.copy()
    if len(comp):
        comp = comp[comp["target"].eq("hydrate_saturation")].sort_values("rmse").head(14)
        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        labels = comp["feature_set"] + "\n" + comp["model"]
        ax.bar(np.arange(len(comp)), comp["rmse"])
        ax.set_xticks(np.arange(len(comp)))
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("External transfer RMSE")
        ax.set_title("External transfer metrics shown for transparency, not selection")
        ax.grid(True, axis="y", alpha=0.25)
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

    # 4. Bin-level error for selected hydrate model.
    hyd_bins = bin_metrics_df[bin_metrics_df["target"].eq("hydrate_saturation")].copy() if len(bin_metrics_df) else pd.DataFrame()
    hyd_bins = hyd_bins[~hyd_bins["reference_bin"].str.contains("threshold", na=False)] if len(hyd_bins) else hyd_bins
    if len(hyd_bins):
        fig, ax = plt.subplots(figsize=(7.5, 4.8))
        ax.bar(np.arange(len(hyd_bins)), hyd_bins["rmse"])
        ax.set_xticks(np.arange(len(hyd_bins)))
        ax.set_xticklabels(hyd_bins["reference_bin"], rotation=35, ha="right")
        ax.set_ylabel("RMSE")
        ax.set_title("Selected hydrate model error by saturation bin")
        ax.grid(True, axis="y", alpha=0.25)
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

    # 5. Rule-based occurrence screen counts.
    fig, ax = plt.subplots(figsize=(8, 5))
    occ_plot = occ_counts.copy()
    for label, sub in occ_plot.groupby("hydrate_occurrence_screen"):
        ax.bar(sub["well_alias"], sub["rows"], label=label, alpha=0.75)
    ax.set_ylabel("Rows")
    ax.set_title("Rule-based hydrate occurrence screen counts")
    ax.legend(fontsize=8)
    ax.grid(True, axis="y", alpha=0.25)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # 6. Occurrence classifier probability by depth for selected model.
    selected_occ = occurrence_result["selected_model_name"]
    occ_pred_plot = occurrence_result["selected_predictions"].copy()
    if len(occ_pred_plot):
        for well_alias, well_df in occ_pred_plot.groupby("well_alias"):
            fig, ax = plt.subplots(figsize=(7, 7))
            ax.plot(well_df["occurrence_probability_ml"], well_df["depth_m"], label="ML occurrence probability", linewidth=1.4)
            ax.scatter(well_df["reference"], well_df["depth_m"], s=10, alpha=0.5, label="S_h-derived label")
            ax.set_xlim(0, 1)
            ax.invert_yaxis()
            ax.set_xlabel("Occurrence probability / label")
            ax.set_ylabel("Depth (m)")
            ax.set_title(f"Occurrence classifier — transfer {well_alias} ({selected_occ})")
            ax.legend()
            ax.grid(True, alpha=0.25)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

# Write compact Excel workbook. If Excel has the file open, write timestamp fallback.
def write_excel(path: Path) -> Path:
    sheets = {
        "summary": summary_df,
        "active_well_policy": active_well_policy_df,
        "header_contracts": header_contract_summary_df,
        "density_phi_policy": density_porosity_policy_summary,
        "chong_core_presence": chong_core_feature_presence_by_well,
        "feature_policy": feature_policy_summary_df,
        "correctness_checks": correctness_checks_df,
        "selected_transfer_metrics": selected_summary_df,
        "metrics_transfer_all": metrics_df.sort_values(["target", "selection_stage", "rmse"], na_position="last"),
        "transfer_by_well": transfer_metrics_by_well_df,
        "cv_model_selection": cv_metrics_df,
        "blind_bin_metrics": bin_metrics_df,
        "occurrence_ml_metrics": occurrence_metrics_df,
        "equations_used": equations_compact,
        "archie_metadata": archie_meta_df,
        "feature_sets": feature_set_catalog_df,
        "feature_importance": feature_importance_df,
        "occurrence_screen": occ_pivot,
        "occurrence_label_rule": pd.concat([occurrence_label_rule_df, occ_label_counts.rename(columns={"occurrence_label_status": "rule", "well_alias": "item"}).assign(purpose="label count by well")], ignore_index=True, sort=False),
        "occurrence_ml_preds": occurrence_predictions_df,
        "selected_predictions": selected_predictions_df,
        "qc_summary": qc_summary,
        "well_summary": well_summary,
        "field_mapping": field_mapping,
    }
    optional_chong_sheets = {
        "chong_ann_protocol": globals().get("chong_ann_protocol_df", pd.DataFrame()),
        "chong_wlc_catalog": globals().get("chong_ann_wlc_catalog_df", pd.DataFrame()),
        "chong_hparam": globals().get("chong_ann_hparam_sensitivity_df", pd.DataFrame()),
        "chong_wlc_summary": globals().get("chong_ann_wlc_summary_df", pd.DataFrame()),
        "chong_pearson": globals().get("chong_ann_pearson_df", pd.DataFrame()),
    }
    if "chong_ann_realization_metrics_df" in globals():
        optional_chong_sheets["chong_repeats_head"] = chong_ann_realization_metrics_df.head(5000)
    sheets.update({k: v for k, v in optional_chong_sheets.items() if isinstance(v, pd.DataFrame) and len(v.columns) > 0})
    try:
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            for sheet_name, df_sheet in sheets.items():
                df_sheet.to_excel(writer, sheet_name=sheet_name[:31], index=False)
        return path
    except PermissionError:
        fallback = path.with_name(f"{path.stem}_{RUN_ID}{path.suffix}")
        with pd.ExcelWriter(fallback, engine="openpyxl") as writer:
            for sheet_name, df_sheet in sheets.items():
                df_sheet.to_excel(writer, sheet_name=sheet_name[:31], index=False)
        return fallback

actual_xlsx = write_excel(OUTPUT_XLSX)

# Save selected models and config.
model_bundle = {
    "run_id": RUN_ID,
    "code_version": CODE_VERSION,
    "well_metadata": WELL_METADATA,
    "active_project_wells": ACTIVE_PROJECT_WELLS,
    "active_well_scope_note": ACTIVE_WELL_SCOPE_NOTE,
    "header_contracts": HEADER_CONTRACTS,
    "source_header_evidence_note": SOURCE_HEADER_EVIDENCE_NOTE,
    "density_porosity_source_policy": DENSITY_POROSITY_SOURCE_POLICY,
    "normalized_input_mode": NORMALIZED_INPUT_MODE,
    "primary_feature_policy": PRIMARY_FEATURE_POLICY,
    "training_selection_mode": TRAINING_SELECTION_MODE,
    "calibration_train_well": CALIBRATION_TRAIN_WELL,
    "transfer_test_wells": TRANSFER_TEST_WELLS,
    "allow_proxy_equation_features_in_primary_model": ALLOW_PROXY_EQUATION_FEATURES_IN_PRIMARY_MODEL,
    "allow_pressure_context_in_primary_model": ALLOW_PRESSURE_CONTEXT_IN_PRIMARY_MODEL,
    "chong_runtime_mode": CHONG_RUNTIME_MODE,
    "chong_ann_realizations": CHONG_ANN_REALIZATIONS,
    "chong_fixed_ann_params": CHONG_FIXED_ANN_PARAMS,
    "deep_resistivity_alias_confirmed": DEEP_RESISTIVITY_ALIAS_CONFIRMED,
    "occurrence_label_rule": {
        "positive_sh_threshold": OCCURRENCE_POSITIVE_SH_THRESHOLD,
        "negative_sh_threshold": OCCURRENCE_NEGATIVE_SH_THRESHOLD,
        "gray_zone_omitted": True,
    },
    "selected_models": {
        r["target"]: {
            "task_type": r["task_type"],
            "selected_model_name": r["selected_model_name"],
            "feature_set": r["feature_set"],
            "features": r["features"],
            "final_pipeline": r["final_pipeline"],
            "validation_pipeline": r["selected_validation_pipeline"],
            "train_wells": r["train_wells"],
            "validation_wells": r.get("validation_wells", []),
            "diagnostic_only": r.get("diagnostic_only", False),
        }
        for r in results + [occurrence_result]
    },
}
joblib.dump(model_bundle, MODEL_JOBLIB)

manifest = {
    "run_id": RUN_ID,
    "code_version": CODE_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_dir": str(INPUT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "model_dir": str(MODEL_DIR),
    "model_results_xlsx": str(actual_xlsx),
    "predictions_csv": str(PREDICTIONS_CSV),
    "selected_predictions_csv": str(SELECTED_PREDICTIONS_CSV),
    "occurrence_predictions_csv": str(OCCURRENCE_PREDICTIONS_CSV),
    "paper_figures_pdf": str(FIGURES_PDF),
    "selected_models_joblib": str(MODEL_JOBLIB),
    "active_well_policy_csv": str(ACTIVE_WELL_POLICY_CSV),
    "header_contract_summary_csv": str(HEADER_CONTRACT_SUMMARY_CSV),
    "density_porosity_policy_csv": str(DENSITY_POROSITY_POLICY_CSV),
    "chong_core_feature_presence_csv": str(CHONG_CORE_FEATURE_PRESENCE_CSV),
    "feature_policy_summary_csv": str(FEATURE_POLICY_SUMMARY_CSV),
    "chong_ann_results_xlsx": str(globals().get("CHONG_ANN_RESULTS_XLSX", "")),
    "chong_ann_wlc_summary_csv": str(globals().get("CHONG_ANN_WLC_SUMMARY_CSV", "")),
    "chong_ann_realization_metrics_csv": str(globals().get("CHONG_ANN_REALIZATION_METRICS_CSV", "")),
    "chong_ann_selected_predictions_csv": str(globals().get("CHONG_ANN_SELECTED_PREDICTIONS_CSV", "")),
    "active_project_wells": ACTIVE_PROJECT_WELLS,
    "active_well_scope_note": ACTIVE_WELL_SCOPE_NOTE,
    "header_contracts": HEADER_CONTRACTS,
    "source_header_evidence_note": SOURCE_HEADER_EVIDENCE_NOTE,
    "density_porosity_source_policy": DENSITY_POROSITY_SOURCE_POLICY,
    "normalized_input_mode": NORMALIZED_INPUT_MODE,
    "primary_feature_policy": PRIMARY_FEATURE_POLICY,
    "training_selection_mode": TRAINING_SELECTION_MODE,
    "calibration_train_well": CALIBRATION_TRAIN_WELL,
    "transfer_test_wells": TRANSFER_TEST_WELLS,
    "allow_proxy_equation_features_in_primary_model": ALLOW_PROXY_EQUATION_FEATURES_IN_PRIMARY_MODEL,
    "hydrate_split": {"train_wells": HYDRATE_TRAIN_WELLS, "validation_wells": HYDRATE_VALIDATION_WELLS},
    "occurrence_split": {"train_wells": OCCURRENCE_TRAIN_WELLS, "validation_wells": OCCURRENCE_VALIDATION_WELLS},
    "water_split": {"train_wells": WATER_TRAIN_WELLS, "validation_wells": WATER_VALIDATION_WELLS, "mode": WATER_MODEL_MODE},
    "occurrence_label_rule": {
        "positive_sh_threshold": OCCURRENCE_POSITIVE_SH_THRESHOLD,
        "negative_sh_threshold": OCCURRENCE_NEGATIVE_SH_THRESHOLD,
        "classification_threshold": OCCURRENCE_CLASSIFICATION_THRESHOLD,
    },
    "selected_models": selected_summary_df.to_dict(orient="records"),
    "correctness_checks": correctness_checks_df.to_dict(orient="records"),
}
MANIFEST_JSON.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

print("\nDONE - files written:")
for p in [actual_xlsx, PREDICTIONS_CSV, SELECTED_PREDICTIONS_CSV, OCCURRENCE_PREDICTIONS_CSV, ACTIVE_WELL_POLICY_CSV, HEADER_CONTRACT_SUMMARY_CSV, DENSITY_POROSITY_POLICY_CSV, CHONG_CORE_FEATURE_PRESENCE_CSV, FEATURE_POLICY_SUMMARY_CSV, FIGURES_PDF, MANIFEST_JSON, MODEL_JOBLIB]:
    print(" -", p, "exists=", p.exists(), "modified=", datetime.fromtimestamp(p.stat().st_mtime) if p.exists() else None)
